imports

In [2]:
%load_ext autoreload
%autoreload 2

import warnings
import pandas as pd
import numpy as np
import plotly.express as px
from gencost.crosswalk import Crosswalk
from gencost.waterfall import DataBySubplant

warnings.simplefilter(action="once")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


create data source object

In [3]:
xwalk = Crosswalk()
# self to be able to copy / paste from waterfall.py for dev ease
self = DataBySubplant(xwalk)

historical data

In [412]:
hist = self.get_exa_by_generator()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [414]:
hist.query('plant_id_eia == 8',engine='python')

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj
389,8,10,2006-01-01,NaN,4.678691e+07,NaN,NaN,NaN,NaN,NaN,...,49.166324,15.915127,-4.924752,NaN,2,<NA>,2,2,1.050357,13.40423
390,8,10,2007-01-01,NaN,4.161849e+07,NaN,NaN,NaN,NaN,NaN,...,49.166324,14.915811,-3.925436,NaN,2,<NA>,2,2,1.057435,14.461664
391,8,10,2008-01-01,NaN,4.364463e+07,NaN,NaN,NaN,NaN,NaN,...,49.166324,13.916496,-2.926121,NaN,2,<NA>,2,2,0.99812,15.459785
392,8,10,2009-01-01,NaN,3.856048e+07,NaN,NaN,NaN,NaN,41736.364147,...,49.166324,12.914442,-1.924067,NaN,2,<NA>,2,2,0.97293,16.432715
393,8,10,2010-01-01,NaN,4.672740e+07,NaN,NaN,NaN,NaN,38853.150624,...,49.166324,11.915127,-0.924752,NaN,2,<NA>,2,2,0.941299,17.374014
394,8,10,2011-01-01,NaN,3.272167e+07,NaN,NaN,NaN,NaN,94971.312151,...,49.166324,10.915811,0.074564,NaN,2,<NA>,2,2,0.973222,18.347236
395,8,10,2012-01-01,NaN,2.186992e+07,NaN,NaN,NaN,NaN,60951.750161,...,49.166324,9.916496,1.073879,NaN,2,<NA>,2,2,0.983656,19.330892
396,8,10,2013-01-01,NaN,2.620028e+07,NaN,NaN,NaN,NaN,41079.481263,...,49.166324,8.914442,2.075933,NaN,2,<NA>,2,2,0.951011,20.281903
397,8,10,2014-01-01,NaN,4.118150e+07,NaN,NaN,NaN,NaN,49907.704881,...,49.166324,7.915127,3.075248,NaN,2,<NA>,2,2,0.935684,21.217587
398,8,10,2015-01-01,NaN,3.749261e+07,NaN,NaN,NaN,NaN,92158.776051,...,49.166324,6.915811,4.074564,NaN,2,<NA>,2,2,0.973045,22.190632


task 1: find instances when a generator reports in the middle of a range

In [194]:
# count n years

# figure out which years are missing

#make list of years belonging to each gen

# compare to list of years we need 


import datetime as dt


df = (hist
.assign(n_years_gen = lambda x: x.groupby(['plant_id_eia','generator_id'])['report_date'].transform('nunique'),
report_year =lambda x: x.report_date.dt.year)
.query('n_years_gen != 15')
.groupby(['plant_id_eia','generator_id'])['report_year'].agg(list).reset_index()
.assign(needed_years = lambda x: x.apply(lambda _: [*range(2006,2020,1)],axis=1),
missing_years = lambda x: (x['needed_years'].map(set) - x['report_year'].map(set)),
years_to_fill_in = lambda x: x['missing_years'].apply(list)))




task 2: find instances when generator changes fuel type

1) establish single fuel definition threshold
2) throw out multi-fuel generator observations
3) do fuel change check somehow - make list? 

In [230]:
#list of cols we need for melt
fuel_consump_cols = ['biofuel_mmbtu',
       'coal_mmbtu', 'natural_gas_mmbtu', 'nuclear_mmbtu', 'other_mmbtu',
       'other_gas_mmbtu', 'petroleum_mmbtu', 'petroleum_coke_mmbtu',
       'renew_mmbtu'] 
#melt to facilitate group by and percent of net gen totals
test = (hist.melt(id_vars=['plant_id_eia','generator_id','report_date'],value_vars=fuel_consump_cols,var_name='fuel_mmbtu',value_name='fuel_consumption')
.assign(
percent_of_gen = lambda x: (x['fuel_consumption'] / x.groupby(['plant_id_eia','generator_id','report_date'])['fuel_consumption'].transform('sum')),
single_fuel = lambda x: np.where(x['percent_of_gen'] >= .9,1,0),
single_fuel_present = lambda x: x.groupby(['plant_id_eia','generator_id','report_date'])['single_fuel'].transform('sum'))
.query('single_fuel_present == 1 & percent_of_gen >= .9')
.assign(n_fuels = lambda x: x.groupby(['plant_id_eia','generator_id'])['fuel_mmbtu'].transform('nunique'))
.query('n_fuels > 1')
#.reset_index()
)

In [231]:
test

,plant_id_eia,generator_id,report_date,fuel_mmbtu,fuel_consumption,percent_of_gen,single_fuel,single_fuel_present,n_fuels
33659,7527,2,2012-01-01,biofuel_mmbtu,323518.0,0.998774,1,1,2
40048,10333,GEN1,2013-01-01,biofuel_mmbtu,450933.0,0.970638,1,1,2
40049,10333,GEN1,2014-01-01,biofuel_mmbtu,1518064.0,0.993868,1,1,2
40050,10333,GEN1,2015-01-01,biofuel_mmbtu,5733380.0,0.983817,1,1,2
40051,10333,GEN1,2016-01-01,biofuel_mmbtu,5104054.0,0.994192,1,1,2
...,...,...,...,...,...,...,...,...,...
545459,6043,8,2016-01-01,renew_mmbtu,695179.0,1.000000,1,1,2
545460,6043,8,2017-01-01,renew_mmbtu,199074.0,1.000000,1,1,2
545461,6043,8,2018-01-01,renew_mmbtu,468365.0,1.000000,1,1,2
545462,6043,8,2019-01-01,renew_mmbtu,249054.0,1.000000,1,1,2


three fuel instances

In [157]:
test.query('n_fuels == 3')['plant_id_eia'].unique()

<IntegerArray>
[3264, 3796, 7242]
Length: 3, dtype: Int64

In [166]:
test.query('n_fuels == 3 & plant_id_eia == 3264',engine='python')

,plant_id_eia,generator_id,report_date,fuel_mwh,net_generation,percent_of_gen,single_fuel,single_fuel_present,n_fuels
86577,3264,3,2006-01-01,coal_net_mwh,609356.901408,1.000000,1,1,3
86578,3264,3,2007-01-01,coal_net_mwh,738759.263239,1.000000,1,1,3
86579,3264,3,2008-01-01,coal_net_mwh,473466.009819,1.000000,1,1,3
86580,3264,3,2009-01-01,coal_net_mwh,293646.001498,0.990453,1,1,3
86581,3264,3,2010-01-01,coal_net_mwh,571956.150823,0.995342,1,1,3
86582,3264,3,2011-01-01,coal_net_mwh,385133.109790,0.994534,1,1,3
86583,3264,3,2012-01-01,coal_net_mwh,79153.068138,0.978164,1,1,3
86584,3264,3,2013-01-01,coal_net_mwh,30665.057444,0.997786,1,1,3
151238,3264,3,2015-01-01,natural_gas_net_mwh,90194.000000,0.980391,1,1,3
151239,3264,3,2016-01-01,natural_gas_net_mwh,221407.679000,0.997357,1,1,3


In [156]:
test.query('n_fuels == 3 & plant_id_eia == 7242',engine='python')

,plant_id_eia,generator_id,report_date,fuel_mwh,net_generation,percent_of_gen,single_fuel,single_fuel_present,n_fuels
97030,7242,1CA,2007-01-01,coal_net_mwh,1.429163e+05,0.987823,1,1,3
97031,7242,1CA,2008-01-01,coal_net_mwh,1.436358e+06,0.980022,1,1,3
97032,7242,1CA,2009-01-01,coal_net_mwh,1.288468e+06,0.983933,1,1,3
97035,7242,1CA,2012-01-01,coal_net_mwh,1.161898e+06,0.988891,1,1,3
97036,7242,1CA,2013-01-01,coal_net_mwh,3.342555e+05,0.981831,1,1,3
97037,7242,1CA,2014-01-01,coal_net_mwh,6.680525e+05,0.917926,1,1,3
97044,7242,1CT,2006-01-01,coal_net_mwh,1.546263e+06,0.973204,1,1,3
97045,7242,1CT,2007-01-01,coal_net_mwh,1.420597e+06,0.979028,1,1,3
97051,7242,1CT,2013-01-01,coal_net_mwh,8.912436e+05,0.943200,1,1,3
161694,7242,1CA,2019-01-01,natural_gas_net_mwh,1.893284e+05,1.000000,1,1,3


In [158]:
test.query('n_fuels == 3 & plant_id_eia == 3796',engine='python')

,plant_id_eia,generator_id,report_date,fuel_mwh,net_generation,percent_of_gen,single_fuel,single_fuel_present,n_fuels
90358,3796,3,2008-01-01,coal_net_mwh,315705.609876,1.000000,1,1,3
90359,3796,3,2009-01-01,coal_net_mwh,174220.286390,0.996273,1,1,3
90360,3796,3,2010-01-01,coal_net_mwh,229964.775925,0.996347,1,1,3
90361,3796,3,2011-01-01,coal_net_mwh,112236.735078,0.994240,1,1,3
90362,3796,3,2012-01-01,coal_net_mwh,60727.931244,0.989780,1,1,3
90363,3796,3,2013-01-01,coal_net_mwh,59908.877294,0.991147,1,1,3
90371,3796,4,2008-01-01,coal_net_mwh,876224.251124,1.000000,1,1,3
90372,3796,4,2009-01-01,coal_net_mwh,744548.834610,0.997052,1,1,3
90373,3796,4,2010-01-01,coal_net_mwh,750688.642075,0.996069,1,1,3
90374,3796,4,2011-01-01,coal_net_mwh,701831.194922,0.995408,1,1,3


In [161]:
test['plant_gen_id'] = str(test['plant_id_eia']) + "_" + test['generator_id']

test['plant_gen_id'].nunique()

44

In [168]:
test

,plant_id_eia,generator_id,report_date,fuel_mwh,net_generation,percent_of_gen,single_fuel,single_fuel_present,n_fuels
40048,10333,GEN1,2013-01-01,biofuel_net_mwh,23481.872,0.973030,1,1,2
40049,10333,GEN1,2014-01-01,biofuel_net_mwh,106943.383,0.993935,1,1,2
40050,10333,GEN1,2015-01-01,biofuel_net_mwh,200282.936,0.982179,1,1,2
40051,10333,GEN1,2016-01-01,biofuel_net_mwh,334718.961,0.994143,1,1,2
40187,10382,GEN1,2015-01-01,biofuel_net_mwh,20109.000,1.000000,1,1,2
...,...,...,...,...,...,...,...,...,...
428442,10633,ST1,2014-01-01,petroleum_net_mwh,12074.770,1.000000,1,1,2
430582,50966,1,2018-01-01,petroleum_net_mwh,37525.124,0.943032,1,1,2
430881,52026,GEN3,2010-01-01,petroleum_net_mwh,1959.722,0.984785,1,1,2
441570,55349,1,2009-01-01,petroleum_net_mwh,5932.500,0.977992,1,1,2


scenario 3: where there are zeros for key columns

my interpretation: when 

In [167]:
hist

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
51,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,1.0,51.915127,67.830253,15.915127,13.739177,2.92732,2,<NA>,2,2
52,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,1.0,52.914442,67.830253,14.915811,14.738493,2.92732,2,<NA>,2,2
53,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,1.0,53.913758,67.830253,13.916496,15.737808,2.92732,2,<NA>,2,2
54,3,1,2009-01-01,NaN,2.274497e+06,7.626184e+03,NaN,NaN,NaN,NaN,...,1.0,54.915811,67.830253,12.914442,16.739862,2.92732,2,<NA>,2,2
55,3,1,2010-01-01,NaN,4.481317e+06,4.298272e+04,NaN,NaN,NaN,NaN,...,1.0,55.915127,67.830253,11.915127,17.739177,2.92732,2,<NA>,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415169,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,1.0,-0.328542,2.587269,2.915811,-15.181727,NaN,454,<NA>,454,ETR
415170,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,1.0,0.670773,2.587269,1.916496,-14.182411,NaN,454,<NA>,454,ETR
415173,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.0,-0.164271,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR
415176,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.0,-0.164271,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR


think about how to miss filling in missing years

In [183]:
df

,plant_id_eia,generator_id,report_year,needed_years,missing_years,years_to_fill_in
0,3,3,"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2019, 2015}","[2016, 2017, 2018, 2019, 2015]"
1,3,A1CT2,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2008, 2006, 2007}","[2008, 2006, 2007]"
2,8,10,"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...",{2019},[2019]
3,8,6,"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2019, 2015}","[2016, 2017, 2018, 2019, 2015]"
4,8,7,"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2019, 2015}","[2016, 2017, 2018, 2019, 2015]"
...,...,...,...,...,...,...
2153,60926,1B,"[2019, 2020]","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2006, 2007, 2008, 2009, 201...","[2016, 2017, 2018, 2006, 2007, 2008, 2009, 201..."
2154,60926,1C,"[2019, 2020]","[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2006, 2007, 2008, 2009, 201...","[2016, 2017, 2018, 2006, 2007, 2008, 2009, 201..."
2155,60927,1A,[2020],"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2019, 2006, 2007, 2008, 200...","[2016, 2017, 2018, 2019, 2006, 2007, 2008, 200..."
2156,60927,1B,[2020],"[2006, 2007, 2008, 2009, 2010, 2011, 2012, 201...","{2016, 2017, 2018, 2019, 2006, 2007, 2008, 200...","[2016, 2017, 2018, 2019, 2006, 2007, 2008, 200..."


In [195]:
split = pd.DataFrame([pd.Series(x) for x in df.years_to_fill_in])
split.columns = ['year_{}'.format(x+1) for x in split.columns]

df = pd.concat([df,split],axis=1)

/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_66186/1588057529.py:1: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  split = pd.DataFrame([pd.Series(x) for x in df.years_to_fill_in])


In [233]:
missing_years_clean = df.melt(
                id_vars=["plant_id_eia", "generator_id"],
                value_vars=[
                    "year_1",
                    "year_2",
                    "year_3",
                    "year_4",
                    "year_5",
                    "year_6",
                    "year_7",
                    "year_8",
                    "year_9",
                    "year_10",
                    "year_11",
                    "year_12",
                    "year_13",
                    "year_14",
                ],
                value_name="year",
            ).query("year.notnull()").assign(year=lambda x: x["year"].astype("Int64"),
fuss = lambda x: 'missing_years')

In [215]:
(df.melt(id_vars=['plant_id_eia','generator_id'],value_vars=['year_1', 'year_2', 'year_3',
       'year_4', 'year_5', 'year_6', 'year_7', 'year_8', 'year_9', 'year_10',
       'year_11', 'year_12', 'year_13', 'year_14'],value_name='year').assign(year = lambda x: x['year'].astype('Int64')
       ).query('year.notnull()')
       .merge(self.xwalk[['plant_id_eia','generator_id','prime_mover','fuel_group']],on=['plant_id_eia','generator_id'],how='left',validate='m:m',indicator=True)
)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


,plant_id_eia,generator_id,variable,year,prime_mover,fuel_group,_merge
0,3,3,year_1,2016,ST,coal,both
1,3,3,year_1,2016,ST,natural_gas,both
2,3,A1CT2,year_1,2008,CC,natural_gas,both
3,8,10,year_1,2019,ST,coal,both
4,8,6,year_1,2016,ST,coal,both
...,...,...,...,...,...,...,...
23201,57185,U005,year_14,2015,CC,natural_gas,both
23202,57185,U006,year_14,2015,CC,natural_gas,both
23203,60927,1A,year_14,2015,CC,natural_gas,both
23204,60927,1B,year_14,2015,CC,natural_gas,both


In [218]:
self.get_gf923_by_generator(counterfactuals=True)

,report_date,plant_id_eia,generator_id,prime_mover_code,energy_source_code,energy_source_code_num,net_mwh,mmbtu,fuel_consumed_for_electricity_mmbtu,fuel_group
1222342,2001-01-01,2,1,HY,WAT,energy_source_code_1,18918.000000,195479.690000,195479.690000,renew
1222606,2001-01-01,3,1,ST,BIT,energy_source_code_1,6371.709286,61866.342266,61866.342266,coal
1222611,2001-01-01,3,1,ST,NG,energy_source_code_2,NaN,NaN,NaN,natural_gas
1222607,2001-01-01,3,2,ST,BIT,energy_source_code_1,6371.709286,61866.342266,61866.342266,coal
1222612,2001-01-01,3,2,ST,NG,energy_source_code_2,NaN,NaN,NaN,natural_gas
...,...,...,...,...,...,...,...,...,...,...
6808425,2022-12-01,65824,VSPRC,PV,SUN,energy_source_code_1,NaN,NaN,NaN,renew
6808437,2022-12-01,65831,LNSTR,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other
6808461,2022-12-01,65836,TOYAH,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other
6808485,2022-12-01,65838,HOEFS,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other


In [219]:
self.get_gf923_by_generator()

,plant_id_eia,generator_id,report_date,SG_mmbtu,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,...,SG_net_mwh,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh
0,1,1,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,827.4,NaN,NaN
1,1,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
2,1,1,2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,818.7,NaN,NaN
3,1,1,2022-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN
4,1,2,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,827.4,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432947,65836,TOYAH,2021-01-01,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
432948,65836,TOYAH,2022-01-01,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
432949,65838,HOEFS,2021-01-01,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
432950,65838,HOEFS,2022-01-01,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [222]:
self.get_exa_by_generator()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,gross_generation_mwh,cems_923_merge,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh
0,1,1,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,8680.2,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1,2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,10647.0,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,1,2022-01-01,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,2,2019-01-01,NaN,NaN,NaN,NaN,NaN,NaN,8680.2,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432947,65836,TOYAH,2021-01-01,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
432948,65836,TOYAH,2022-01-01,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
432949,65838,HOEFS,2021-01-01,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
432950,65838,HOEFS,2022-01-01,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,NaN,left_only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [220]:
self.get_cems_by_generator()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


,plant_id_eia,generator_id,report_date,generator_starts,fuel_starts,gross_generation_mwh
51,3,1,2006-01-01,7.0,7.0,1123980.0
52,3,1,2007-01-01,13.0,13.0,1028301.0
53,3,1,2008-01-01,15.0,15.0,928095.0
54,3,1,2009-01-01,44.0,44.0,243753.0
55,3,1,2010-01-01,14.0,14.0,469201.0
...,...,...,...,...,...,...
415104,60926,1B,2020-01-01,36.0,37.0,2997852.5
415105,60926,1C,2020-01-01,36.0,37.0,0.0
415112,60927,1A,2020-01-01,51.0,54.0,1663118.5
415113,60927,1B,2020-01-01,51.0,54.0,1663118.5


In [232]:
self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

,plant_id_eia,generator_id,report_date,mmbtu,fuel_consumption,percent_of_gen,single_fuel,single_fuel_present,n_fuels
33659,7527,2,2012-01-01,biofuel_mmbtu,323518.0,0.998774,1,1,2
40048,10333,GEN1,2013-01-01,biofuel_mmbtu,450933.0,0.970638,1,1,2
40049,10333,GEN1,2014-01-01,biofuel_mmbtu,1518064.0,0.993868,1,1,2
40050,10333,GEN1,2015-01-01,biofuel_mmbtu,5733380.0,0.983817,1,1,2
40051,10333,GEN1,2016-01-01,biofuel_mmbtu,5104054.0,0.994192,1,1,2
...,...,...,...,...,...,...,...,...,...
545459,6043,8,2016-01-01,renew_mmbtu,695179.0,1.000000,1,1,2
545460,6043,8,2017-01-01,renew_mmbtu,199074.0,1.000000,1,1,2
545461,6043,8,2018-01-01,renew_mmbtu,468365.0,1.000000,1,1,2
545462,6043,8,2019-01-01,renew_mmbtu,249054.0,1.000000,1,1,2


boneyard 

what we took off year exploded df missing years 

 .merge(
                self.xwalk[
                    ["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]
                ],
                on=["plant_id_eia", "generator_id"],
                how="left",
                validate="m:m",
                indicator=True,
            )
        )


In [234]:
missing_years_clean

,plant_id_eia,generator_id,variable,year,fuss
0,3,3,year_1,2016,missing_years
1,3,A1CT2,year_1,2008,missing_years
2,8,10,year_1,2019,missing_years
3,8,6,year_1,2016,missing_years
4,8,7,year_1,2016,missing_years
...,...,...,...,...,...
30013,57185,U005,year_14,2015,missing_years
30014,57185,U006,year_14,2015,missing_years
30209,60927,1A,year_14,2015,missing_years
30210,60927,1B,year_14,2015,missing_years


In [239]:
sorted = test.sort_values(by=['plant_id_eia','generator_id','report_date'],ascending=True)

In [241]:
sorted['fuel'] = sorted.groupby(['plant_id_eia','generator_id'])['fuel_mmbtu'].transform('last')

In [242]:
sorted.query('fuel !=')

,plant_id_eia,generator_id,report_date,fuel_mmbtu,fuel_consumption,percent_of_gen,single_fuel,single_fuel_present,n_fuels,fuel
64652,3,1,2006-01-01,coal_mmbtu,1.008871e+07,0.997083,1,1,2,natural_gas_mmbtu
64653,3,1,2007-01-01,coal_mmbtu,9.807031e+06,1.000000,1,1,2,natural_gas_mmbtu
64654,3,1,2008-01-01,coal_mmbtu,8.176078e+06,1.000000,1,1,2,natural_gas_mmbtu
64655,3,1,2009-01-01,coal_mmbtu,2.274497e+06,0.996658,1,1,2,natural_gas_mmbtu
64656,3,1,2010-01-01,coal_mmbtu,4.481317e+06,0.990500,1,1,2,natural_gas_mmbtu
...,...,...,...,...,...,...,...,...,...,...
186473,55505,BR2,2016-01-01,natural_gas_mmbtu,1.090730e+05,0.990902,1,1,2,natural_gas_mmbtu
186474,55505,BR2,2017-01-01,natural_gas_mmbtu,7.324400e+04,0.990386,1,1,2,natural_gas_mmbtu
186475,55505,BR2,2018-01-01,natural_gas_mmbtu,2.330545e+05,0.992475,1,1,2,natural_gas_mmbtu
186476,55505,BR2,2019-01-01,natural_gas_mmbtu,5.666600e+05,0.991677,1,1,2,natural_gas_mmbtu


In [411]:
df = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [410]:
df

,plant_id_eia,generator_id,variable,year,fuss,report_date,prime_mover_code,fuel_group,prime_mover
0,3,3,year_1,2016,missing_years,NaT,<NA>,<NA>,<NA>
1,3,A1CT2,year_1,2008,missing_years,2019-01-01,CT,natural_gas,CC
2,8,10,year_1,2019,missing_years,2019-01-01,ST,coal,ST
3,8,10,year_1,2019,missing_years,2019-01-01,ST,petroleum,ST
4,8,6,year_1,2016,missing_years,NaT,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...
13617,57185,U005,year_14,2015,missing_years,NaT,<NA>,<NA>,<NA>
13618,57185,U006,year_14,2015,missing_years,NaT,<NA>,<NA>,<NA>
13619,60927,1A,year_14,2015,missing_years,NaT,<NA>,<NA>,<NA>
13620,60927,1B,year_14,2015,missing_years,NaT,<NA>,<NA>,<NA>


In [415]:
df_gf23 = self.get_gf923_by_generator(counterfactuals=True)

In [417]:
df_gf23.query('plant_id_eia == 8 & generator_id == "10"',engine='python')

,report_date,plant_id_eia,generator_id,prime_mover_code,energy_source_code,energy_source_code_num,net_mwh,mmbtu,fuel_consumed_for_electricity_mmbtu,fuel_group
1225634,2001-01-01,8,10,ST,BIT,energy_source_code_1,393568.717160,3.847835e+06,3.847835e+06,coal
1225639,2001-01-01,8,10,ST,WC,energy_source_code_2,NaN,NaN,NaN,coal
1225640,2001-02-01,8,10,ST,BIT,energy_source_code_1,326051.624197,3.139236e+06,3.139236e+06,coal
1225645,2001-02-01,8,10,ST,WC,energy_source_code_2,NaN,NaN,NaN,coal
1225646,2001-03-01,8,10,ST,BIT,energy_source_code_1,254601.388579,2.555122e+06,2.555122e+06,coal
...,...,...,...,...,...,...,...,...,...,...
1186530,2019-03-01,8,10,ST,DFO,energy_source_code_2,-2.785870,0.000000e+00,0.000000e+00,petroleum
1186533,2019-04-01,8,10,ST,BIT,energy_source_code_1,-510.315216,0.000000e+00,0.000000e+00,coal
1186536,2019-04-01,8,10,ST,DFO,energy_source_code_2,-0.684784,0.000000e+00,0.000000e+00,petroleum
1186539,2019-05-01,8,10,ST,BIT,energy_source_code_1,NaN,0.000000e+00,0.000000e+00,coal


In [418]:
cems = self.get_cems_by_generator()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


In [421]:
cems.query('plant_id_eia == 8 & generator_id == "10"',engine='python')

,plant_id_eia,generator_id,report_date,generator_starts,fuel_starts,gross_generation_mwh
389,8,10,2006-01-01,13.0,9.0,5502690.0
390,8,10,2007-01-01,13.0,13.0,4476716.0
391,8,10,2008-01-01,33.0,33.0,4767016.0
392,8,10,2009-01-01,41.0,41.0,4212659.0
393,8,10,2010-01-01,5.0,5.0,5078736.0
394,8,10,2011-01-01,17.0,17.0,3588795.0
395,8,10,2012-01-01,14.0,14.0,2374948.0
396,8,10,2013-01-01,13.0,13.0,2861300.0
397,8,10,2014-01-01,16.0,16.0,4560601.0
398,8,10,2015-01-01,15.0,15.0,4178165.0


In [422]:
df860 = self.get_860_by_x(subplant_id_col='generator_id')

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [423]:
df860.query('plant_id_eia == 8 & generator_id == "10"',engine='python')

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,...,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
16,195,SOCO,AL,8,10,2001-01-01,788.8,ST,1,1,...,1,28.251882,49.166324,20.914442,-9.924067,NaN,2,<NA>,2,2
15875,195,SOCO,AL,8,10,2002-01-01,788.8,ST,1,1,...,1,29.251198,49.166324,19.915127,-8.924752,NaN,2,<NA>,2,2
32287,195,SOCO,AL,8,10,2003-01-01,788.8,ST,1,1,...,1,30.250513,49.166324,18.915811,-7.925436,NaN,2,<NA>,2,2
49043,195,SOCO,AL,8,10,2004-01-01,788.8,ST,1,1,...,1,31.249829,49.166324,17.916496,-6.926121,NaN,2,<NA>,2,2
65813,195,SOCO,AL,8,10,2005-01-01,788.8,ST,1,1,...,1,32.251882,49.166324,16.914442,-5.924067,NaN,2,<NA>,2,2
82620,195,SOCO,AL,8,10,2006-01-01,788.8,ST,1,1,...,1,33.251198,49.166324,15.915127,-4.924752,NaN,2,<NA>,2,2
99544,195,SOCO,AL,8,10,2007-01-01,788.8,ST,1,1,...,1,34.250513,49.166324,14.915811,-3.925436,NaN,2,<NA>,2,2
116885,195,SOCO,AL,8,10,2008-01-01,788.8,ST,1,1,...,1,35.249829,49.166324,13.916496,-2.926121,NaN,2,<NA>,2,2
134544,195,SOCO,AL,8,10,2009-01-01,788.8,ST,1,1,...,1,36.251882,49.166324,12.914442,-1.924067,NaN,2,<NA>,2,2
152420,195,SOCO,AL,8,10,2010-01-01,788.8,ST,1,1,...,1,37.251198,49.166324,11.915127,-0.924752,NaN,2,<NA>,2,2


In [408]:
test = (hist.groupby(['plant_id_eia','generator_id',pd.Grouper(key="report_date", freq="YS"),'prime_mover_code','fuel_group']).agg({'mmbtu':'sum','net_mwh':'sum'})
.reset_index()
.query('report_date <= "2020-01-01"')
.query('mmbtu > 0 & net_mwh > 0')
.sort_values(by=['plant_id_eia','generator_id','report_date'],ascending=True)
.assign(prime_fuel = lambda x: x['prime_mover_code'] + "_" + x['fuel_group'],
year= lambda x: (x['report_date'].dt.year))
#year = lambda x: str(x['report_year']))
.groupby(['plant_id_eia','generator_id','year'])['prime_fuel'].agg(list)
.reset_index()
.pivot(index=['plant_id_eia','generator_id'],
columns=['year'],values=['prime_fuel'])
.rename_axis(None, axis=1).reset_index(drop=True)
)

#aggfunc=lambda x:" ".join(str(v) for v in x))
#aggfunc='first')



#test.columns = map("_".join, test.columns)
#test = test.loc[:, test.sum(axis=0) != 0].reset_index()
#[['plant_id_eia','generator_id','report_date','prime_mover_code','fuel_group']])



TypeError: Must pass list-like as `names`.

In [407]:
test.query('plant_id_eia == 3',engine='python')

,plant_id_eia,generator_id,year,prime_fuel
24,3,1,2001,[ST_coal]
25,3,1,2002,[ST_coal]
26,3,1,2003,"[ST_coal, ST_natural_gas]"
27,3,1,2004,"[ST_coal, ST_natural_gas]"
28,3,1,2005,"[ST_coal, ST_natural_gas]"
...,...,...,...,...
219,3,A2ST,2016,[CA_natural_gas]
220,3,A2ST,2017,[CA_natural_gas]
221,3,A2ST,2018,[CA_natural_gas]
222,3,A2ST,2019,[CA_natural_gas]


In [331]:
missing_years_clean

,plant_id_eia,generator_id,variable,year,fuss
0,3,3,year_1,2016,missing_years
1,3,A1CT2,year_1,2008,missing_years
2,8,10,year_1,2019,missing_years
3,8,6,year_1,2016,missing_years
4,8,7,year_1,2016,missing_years
...,...,...,...,...,...
30013,57185,U005,year_14,2015,missing_years
30014,57185,U006,year_14,2015,missing_years
30209,60927,1A,year_14,2015,missing_years
30210,60927,1B,year_14,2015,missing_years


In [300]:
test.query('plant_id_eia == 3 & generator_id == "1"',engine='python')

,plant_id_eia,generator_id,report_date,prime_mover_code,fuel_group,mmbtu,net_mwh,n_fuels
77,3,1,2020-01-01,ST,natural_gas,2.861272e+04,1554.454351,1
72,3,1,2015-01-01,ST,natural_gas,5.947680e+04,51.994385,1
73,3,1,2016-01-01,ST,natural_gas,1.618925e+05,233.870694,1
74,3,1,2017-01-01,ST,natural_gas,1.966532e+05,1150.019335,1
70,3,1,2014-01-01,ST,coal,2.479372e+05,18975.754113,2
68,3,1,2013-01-01,ST,coal,3.448870e+05,30367.124070,2
75,3,1,2018-01-01,ST,natural_gas,3.908812e+05,2312.719002,1
76,3,1,2019-01-01,ST,natural_gas,4.135949e+05,1952.538969,1
66,3,1,2012-01-01,ST,coal,1.428320e+06,148021.028081,2
60,3,1,2009-01-01,ST,coal,2.274497e+06,216274.279645,2


In [290]:
df = self.get_exa_by_generator()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [293]:
df.to_parquet("historic_data_gen_level_updated_0523.parquet")

In [291]:
df.columns

Index(['plant_id_eia', 'generator_id', 'report_date', 'biofuel_mmbtu',
       'coal_mmbtu', 'natural_gas_mmbtu', 'nuclear_mmbtu', 'other_mmbtu',
       'other_gas_mmbtu', 'petroleum_mmbtu', 'petroleum_coke_mmbtu',
       'renew_mmbtu', 'biofuel_net_mwh', 'coal_net_mwh', 'natural_gas_net_mwh',
       'nuclear_net_mwh', 'other_net_mwh', 'other_gas_net_mwh',
       'petroleum_net_mwh', 'petroleum_coke_net_mwh', 'renew_net_mwh',
       'generator_starts', 'fuel_starts', 'gross_generation_mwh',
       'biofuel_gross_mwh', 'coal_gross_mwh', 'natural_gas_gross_mwh',
       'other_gross_mwh', 'other_gas_gross_mwh', 'petroleum_gross_mwh',
       'petroleum_coke_gross_mwh', 'renew_gross_mwh', 'utility_id_eia',
       'balancing_authority_code_eia', 'state', 'capacity_mw', 'prime_mover',
       'associated_combined_heat_power', 'duct_burners',
       'bypass_heat_recovery', 'solid_fuel_gasification', 'carbon_capture',
       'fluidized_bed_tech', 'pulverized_coal_tech', 'stoker_tech',
       'oth

In [284]:
df860 = self.pudl_tabl.gens_eia860()
df860.columns.to_list()

['report_date',
 'plant_id_eia',
 'plant_id_pudl',
 'plant_name_eia',
 'utility_id_eia',
 'utility_id_pudl',
 'utility_name_eia',
 'generator_id',
 'associated_combined_heat_power',
 'bga_source',
 'bypass_heat_recovery',
 'capacity_mw',
 'carbon_capture',
 'city',
 'cofire_fuels',
 'county',
 'current_planned_generator_operating_date',
 'data_maturity',
 'deliver_power_transgrid',
 'distributed_generation',
 'duct_burners',
 'energy_source_1_transport_1',
 'energy_source_1_transport_2',
 'energy_source_1_transport_3',
 'energy_source_2_transport_1',
 'energy_source_2_transport_2',
 'energy_source_2_transport_3',
 'energy_source_code_1',
 'energy_source_code_2',
 'energy_source_code_3',
 'energy_source_code_4',
 'energy_source_code_5',
 'energy_source_code_6',
 'energy_storage_capacity_mwh',
 'ferc_qualifying_facility',
 'fluidized_bed_tech',
 'fuel_type_code_pudl',
 'fuel_type_count',
 'generator_operating_date',
 'generator_retirement_date',
 'latitude',
 'longitude',
 'minimum_load_

In [425]:
df_gf23

,report_date,plant_id_eia,generator_id,prime_mover_code,energy_source_code,energy_source_code_num,net_mwh,mmbtu,fuel_consumed_for_electricity_mmbtu,fuel_group
1222342,2001-01-01,2,1,HY,WAT,energy_source_code_1,18918.000000,195479.690000,195479.690000,renew
1222606,2001-01-01,3,1,ST,BIT,energy_source_code_1,6371.709286,61866.342266,61866.342266,coal
1222611,2001-01-01,3,1,ST,NG,energy_source_code_2,NaN,NaN,NaN,natural_gas
1222607,2001-01-01,3,2,ST,BIT,energy_source_code_1,6371.709286,61866.342266,61866.342266,coal
1222612,2001-01-01,3,2,ST,NG,energy_source_code_2,NaN,NaN,NaN,natural_gas
...,...,...,...,...,...,...,...,...,...,...
6808425,2022-12-01,65824,VSPRC,PV,SUN,energy_source_code_1,NaN,NaN,NaN,renew
6808437,2022-12-01,65831,LNSTR,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other
6808461,2022-12-01,65836,TOYAH,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other
6808485,2022-12-01,65838,HOEFS,BA,MWH,energy_source_code_1,NaN,NaN,NaN,other


In [426]:
missing_years_clean

,plant_id_eia,generator_id,variable,year,fuss
0,3,3,year_1,2016,missing_years
1,3,A1CT2,year_1,2008,missing_years
2,8,10,year_1,2019,missing_years
3,8,6,year_1,2016,missing_years
4,8,7,year_1,2016,missing_years
...,...,...,...,...,...
30013,57185,U005,year_14,2015,missing_years
30014,57185,U006,year_14,2015,missing_years
30209,60927,1A,year_14,2015,missing_years
30210,60927,1B,year_14,2015,missing_years


In [ ]:
test = (hist.groupby(['plant_id_eia','generator_id',pd.Grouper(key="report_date", freq="YS"),'prime_mover_code','fuel_group']).agg({'mmbtu':'sum','net_mwh':'sum'})
.reset_index()
.query('report_date <= "2020-01-01"')
.query('mmbtu > 0 & net_mwh > 0')
.sort_values(by=['plant_id_eia','generator_id','report_date'],ascending=True)
.assign(prime_fuel = lambda x: x['prime_mover_code'] + "_" + x['fuel_group'],
year= lambda x: (x['report_date'].dt.year))
#year = lambda x: str(x['report_year']))
.groupby(['plant_id_eia','generator_id','year'])['prime_fuel'].agg(list)
.reset_index()
.pivot(index=['plant_id_eia','generator_id'],
columns=['year'],values=['prime_fuel'])
.rename_axis(None, axis=1).reset_index(drop=True)
)

#aggfunc=lambda x:" ".join(str(v) for

figure out which fuel and prime to fill in for 

In [430]:
(df_gf23.groupby(['plant_id_eia','generator_id',pd.Grouper(key="report_date", freq="YS"),'prime_mover_code','fuel_group']).agg({'mmbtu':'sum','net_mwh':'sum'})
.reset_index()
.query('report_date <= "2020-01-01"')
.query('mmbtu > 0 & net_mwh > 0')
.assign(prime_fuel = lambda x: x['prime_mover_code'] + "_" + x['fuel_group'])
.pivot(index=['plant_id_eia','generator_id','report_date'],
columns=['mmbtu'],values=['prime_fuel'])
)

/var/folders/lw/gjwq5pd52hb01x363x3vyw6c0000gp/T/ipykernel_66186/4289876875.py:6: PerformanceWarning: The following operation may generate 2677088634 cells in the resulting pandas object.
  .pivot(index=['plant_id_eia','generator_id','report_date'],


trying to stuff figure out

In [3]:
missing_data = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [9]:
missing_data

,plant_id_eia,generator_id,variable,year,fuss,report_date,prime_mover_code,fuel_group,prime_mover
0,3,3,year_1,2016,missing_years,NaT,<NA>,<NA>,<NA>
1,3,A1CT2,year_1,2008,missing_years,2020-01-01,CT,natural_gas,CC
2,8,10,year_1,2019,missing_years,NaT,<NA>,<NA>,<NA>
3,8,6,year_1,2016,missing_years,NaT,<NA>,<NA>,<NA>
4,8,7,year_1,2016,missing_years,NaT,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...
13381,57185,U005,year_14,2015,missing_years,2020-01-01,CT,natural_gas,CC
13382,57185,U006,year_14,2015,missing_years,2020-01-01,CT,natural_gas,CC
13383,60927,1A,year_14,2015,missing_years,2020-01-01,CT,natural_gas,CC
13384,60927,1B,year_14,2015,missing_years,2020-01-01,CT,natural_gas,CC


In [14]:
df_gf23 = self.get_gf923_by_generator(counterfactuals=True)

In [30]:
check = (df_gf23.groupby(['plant_id_eia','generator_id',pd.Grouper(key="report_date", freq="YS"),'prime_mover_code','fuel_group']).agg({'mmbtu':'sum','net_mwh':'sum'})
.reset_index()
.query('report_date <= "2020-01-01"')
.query('mmbtu > 0 & net_mwh > 0')
.sort_values(by=['plant_id_eia','generator_id','report_date','mmbtu'],ascending=True)
.drop_duplicates(subset=['plant_id_eia','generator_id'],keep='last')
)

In [33]:
check

,plant_id_eia,generator_id,report_date,prime_mover_code,fuel_group,mmbtu,net_mwh
0,1,1,2019-01-01,IC,petroleum,8680.200000,827.400000
4,1,2,2019-01-01,IC,petroleum,8680.200000,827.400000
8,1,3,2019-01-01,IC,petroleum,4822.333333,459.666667
12,1,5,2019-01-01,IC,petroleum,6751.266667,643.533333
16,1,WT1,2019-01-01,WT,renew,4550.500000,511.000000
...,...,...,...,...,...,...,...
550455,64723,50835,2020-01-01,PV,renew,208621.000000,23796.000000
550491,64748,CR18A,2020-01-01,FC,natural_gas,3637.000000,2335.000000
550496,64749,CR18B,2020-01-01,FC,natural_gas,17252.000000,2677.000000
550505,64753,FDX10,2020-01-01,FC,natural_gas,60594.000000,8670.000000


test after 

In [42]:
missing_data = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [84]:
missing_yrs = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [85]:
missing_yrs

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,3,3,2016,missing_years,ST,natural_gas,2,NaN
1,3,3,2016,missing_years,ST,natural_gas,2,NaN
2,3,3,2016,missing_years,ST,natural_gas,2,NaN
3,3,3,2016,missing_years,ST,natural_gas,2,NaN
4,3,3,2016,missing_years,ST,natural_gas,2,NaN
...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585


In [91]:
missing = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [81]:
single_fuel_swtich

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,7527,2,2012,fuel_switch,CC,natural_gas,CAISO,26.168378
1,10333,GEN1,2013,fuel_switch,ST,coal,FPC,33.500342
2,10333,GEN1,2014,fuel_switch,ST,coal,FPC,33.500342
3,10333,GEN1,2015,fuel_switch,ST,coal,FPC,33.500342
4,10333,GEN1,2016,fuel_switch,ST,coal,FPC,33.500342
...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585


In [45]:
xwalk = self.xwalk

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk


In [60]:
test = single_fuel_swtich.merge(xwalk,on=['plant_id_eia','generator_id','fuel_group'],how='left',indicator=True)

In [51]:
test.query('prime_mover.isnull()')['fuel_group'].value_counts(normalize=True)

petroleum      0.797101
natural_gas    0.202899
Name: fuel_group, dtype: float64

In [63]:
test.query('prime_mover.isnull()')['fuel_group'].value_counts()

petroleum      55
natural_gas    14
Name: fuel_group, dtype: int64

In [62]:
test['plant_gen_id'] = str(test['plant_id_eia']) + "_" + test['generator_id']

test.query('prime_mover.isnull()')['plant_gen_id'].nunique()

3

In [65]:
test.query('prime_mover.isnull()')['plant_id_eia'].unique()

<IntegerArray>
[130, 3264, 3399, 6469]
Length: 4, dtype: Int64

In [66]:
test.query('prime_mover.isnull() & plant_id_eia == 130',engine='python')

,plant_id_eia,generator_id,year,fuel_group,fuss,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime,_merge,plant_gen_id
192,130,2,2006,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
193,130,2,2007,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
194,130,2,2008,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
195,130,2,2009,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
196,130,2,2010,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
197,130,2,2011,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
198,130,2,2012,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
199,130,2,2013,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
200,130,2,2014,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
201,130,2,2015,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...


In [67]:
test.query('prime_mover.isnull() & plant_id_eia == 3264',engine='python')

,plant_id_eia,generator_id,year,fuel_group,fuss,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime,_merge,plant_gen_id
1086,3264,3,2006,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1087,3264,3,2007,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1088,3264,3,2008,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1089,3264,3,2009,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1090,3264,3,2010,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1091,3264,3,2011,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1092,3264,3,2012,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1093,3264,3,2013,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
2226,3264,3,2015,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
2227,3264,3,2016,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...


In [69]:
test

,plant_id_eia,generator_id,year,fuel_group,fuss,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime,_merge,plant_gen_id
0,7527,2,2012,natural_gas,fuel_switch,1,0,0.0,CC,1,17.5,1995-10-01,7527,7527_CC_natural_gas,False,both,0 7527\n1 10333\n2 10333\n3...
1,10333,GEN1,2013,coal,fuel_switch,1,0,0.0,ST,1,125.0,1988-06-01,10333,10333_ST_coal,True,both,0 7527\n1 10333\n2 10333\n3...
2,10333,GEN1,2014,coal,fuel_switch,1,0,0.0,ST,1,125.0,1988-06-01,10333,10333_ST_coal,True,both,0 7527\n1 10333\n2 10333\n3...
3,10333,GEN1,2015,coal,fuel_switch,1,0,0.0,ST,1,125.0,1988-06-01,10333,10333_ST_coal,True,both,0 7527\n1 10333\n2 10333\n3...
4,10333,GEN1,2016,coal,fuel_switch,1,0,0.0,ST,1,125.0,1988-06-01,10333,10333_ST_coal,True,both,0 7527\n1 10333\n2 10333\n3...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,petroleum,fuel_switch,BR2,1,0.0,GT,<NA>,77.1,2002-05-01,55505,55505_GT_petroleum,True,both,0 7527\n1 10333\n2 10333\n3...
2393,55505,BR2,2017,petroleum,fuel_switch,BR2,1,0.0,GT,<NA>,77.1,2002-05-01,55505,55505_GT_petroleum,True,both,0 7527\n1 10333\n2 10333\n3...
2394,55505,BR2,2018,petroleum,fuel_switch,BR2,1,0.0,GT,<NA>,77.1,2002-05-01,55505,55505_GT_petroleum,True,both,0 7527\n1 10333\n2 10333\n3...
2395,55505,BR2,2019,petroleum,fuel_switch,BR2,1,0.0,GT,<NA>,77.1,2002-05-01,55505,55505_GT_petroleum,True,both,0 7527\n1 10333\n2 10333\n3...


In [68]:
test.query('prime_mover.isnull() & plant_id_eia == 3399',engine='python')

,plant_id_eia,generator_id,year,fuel_group,fuss,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime,_merge,plant_gen_id
1118,3399,1,2006,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1119,3399,1,2007,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1120,3399,1,2008,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1121,3399,1,2009,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1122,3399,1,2010,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1123,3399,1,2011,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1124,3399,1,2012,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1125,3399,1,2013,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1126,3399,1,2014,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...
1127,3399,1,2015,petroleum,fuel_switch,<NA>,<NA>,NaN,<NA>,<NA>,NaN,NaT,<NA>,NaN,NaN,left_only,0 7527\n1 10333\n2 10333\n3...


In [72]:
hist_data = self.get_exa_by_generator()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [73]:
df_860 = self.get_860_by_x(subplant_id_col='generator_id')

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [90]:
df_860

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,...,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
0,195,SOCO,AL,2,1,2001-01-01,45.0,HY,1,1,...,1,37.505818,58.420260,20.914442,-22.507147,2.92732,2,<NA>,2,2
1,195,SOCO,AL,3,1,2001-01-01,153.1,ST,1,1,...,1,46.915811,67.830253,20.914442,8.739862,2.92732,2,<NA>,2,2
2,195,SOCO,AL,3,2,2001-01-01,153.1,ST,1,1,...,1,46.505133,67.419576,20.914442,8.329184,2.92732,2,<NA>,2,2
3,195,SOCO,AL,3,3,2001-01-01,272.0,ST,1,1,...,1,41.505818,62.420260,20.914442,3.329869,NaN,2,<NA>,2,2
4,195,SOCO,AL,3,4,2001-01-01,403.7,ST,1,1,...,1,31.085558,52.000000,20.914442,-7.090392,2.92732,2,<NA>,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427134,61980,NaN,CA,65824,VSPRC,2022-01-01,1.1,PV,1,1,...,1,1.752225,1.667351,-0.084873,-1.315511,NaN,<NA>,<NA>,<NA>,NaN
427135,65076,ERCO,TX,65831,LNSTR,2022-01-01,9.9,BA,1,1,...,1,NaN,NaN,-0.084873,NaN,NaN,<NA>,<NA>,<NA>,ERCO
427136,65076,ERCO,TX,65836,TOYAH,2022-01-01,10.4,BA,1,1,...,1,0.251882,0.167009,-0.084873,-1.996059,NaN,<NA>,<NA>,<NA>,ERCO
427137,65076,ERCO,TX,65838,HOEFS,2022-01-01,2.0,BA,1,1,...,1,0.670773,0.585900,-0.084873,-1.577168,NaN,<NA>,<NA>,<NA>,ERCO


In [77]:
(single_fuel_swtich
.merge(df_860.assign(year = lambda x: x['report_date'].dt.year)[['plant_id_eia','generator_id','year','final_ba_code','age_in_current_year']],
on=['plant_id_eia','generator_id','year'],
how='left',
indicator=True)
)

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,_merge
0,7527,2,2012,fuel_switch,CC,natural_gas,CAISO,26.168378,both
1,10333,GEN1,2013,fuel_switch,ST,coal,FPC,33.500342,both
2,10333,GEN1,2014,fuel_switch,ST,coal,FPC,33.500342,both
3,10333,GEN1,2015,fuel_switch,ST,coal,FPC,33.500342,both
4,10333,GEN1,2016,fuel_switch,ST,coal,FPC,33.500342,both
...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,both
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,both
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,both
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,both


In [74]:
df_860

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,...,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
0,195,SOCO,AL,2,1,2001-01-01,45.0,HY,1,1,...,1,37.505818,58.420260,20.914442,-22.507147,2.92732,2,<NA>,2,2
1,195,SOCO,AL,3,1,2001-01-01,153.1,ST,1,1,...,1,46.915811,67.830253,20.914442,8.739862,2.92732,2,<NA>,2,2
2,195,SOCO,AL,3,2,2001-01-01,153.1,ST,1,1,...,1,46.505133,67.419576,20.914442,8.329184,2.92732,2,<NA>,2,2
3,195,SOCO,AL,3,3,2001-01-01,272.0,ST,1,1,...,1,41.505818,62.420260,20.914442,3.329869,NaN,2,<NA>,2,2
4,195,SOCO,AL,3,4,2001-01-01,403.7,ST,1,1,...,1,31.085558,52.000000,20.914442,-7.090392,2.92732,2,<NA>,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427134,61980,NaN,CA,65824,VSPRC,2022-01-01,1.1,PV,1,1,...,1,1.752225,1.667351,-0.084873,-1.315511,NaN,<NA>,<NA>,<NA>,NaN
427135,65076,ERCO,TX,65831,LNSTR,2022-01-01,9.9,BA,1,1,...,1,NaN,NaN,-0.084873,NaN,NaN,<NA>,<NA>,<NA>,ERCO
427136,65076,ERCO,TX,65836,TOYAH,2022-01-01,10.4,BA,1,1,...,1,0.251882,0.167009,-0.084873,-1.996059,NaN,<NA>,<NA>,<NA>,ERCO
427137,65076,ERCO,TX,65838,HOEFS,2022-01-01,2.0,BA,1,1,...,1,0.670773,0.585900,-0.084873,-1.577168,NaN,<NA>,<NA>,<NA>,ERCO


In [87]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj
51,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,67.830253,15.915127,13.739177,2.92732,2,<NA>,2,2,1.050357,13.40423
52,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,67.830253,14.915811,14.738493,2.92732,2,<NA>,2,2,1.057435,14.461664
53,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,67.830253,13.916496,15.737808,2.92732,2,<NA>,2,2,0.99812,15.459785
54,3,1,2009-01-01,NaN,2.274497e+06,7.626184e+03,NaN,NaN,NaN,NaN,...,67.830253,12.914442,16.739862,2.92732,2,<NA>,2,2,0.97293,16.432715
55,3,1,2010-01-01,NaN,4.481317e+06,4.298272e+04,NaN,NaN,NaN,NaN,...,67.830253,11.915127,17.739177,2.92732,2,<NA>,2,2,0.941299,17.374014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415169,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,2.587269,2.915811,-15.181727,NaN,454,<NA>,454,ETR,0.923801,24.360724
415170,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,2.587269,1.916496,-14.182411,NaN,454,<NA>,454,ETR,0.964819,25.325543
415173,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR,0.964819,25.325543
415176,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR,0.964819,25.325543


In [88]:
missing_yrs

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,3,3,2016,missing_years,ST,natural_gas,2,NaN
1,3,3,2016,missing_years,ST,natural_gas,2,NaN
2,3,3,2016,missing_years,ST,natural_gas,2,NaN
3,3,3,2016,missing_years,ST,natural_gas,2,NaN
4,3,3,2016,missing_years,ST,natural_gas,2,NaN
...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585


filling in process

In [96]:
missing

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,3,3,2016,missing_years,ST,natural_gas,2,NaN
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN
28,8,10,2019,missing_years,ST,coal,2,NaN
46,8,6,2016,missing_years,ST,coal,2,NaN
60,8,7,2016,missing_years,ST,coal,2,NaN
...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585


In [97]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj
51,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,67.830253,15.915127,13.739177,2.92732,2,<NA>,2,2,1.050357,13.40423
52,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,67.830253,14.915811,14.738493,2.92732,2,<NA>,2,2,1.057435,14.461664
53,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,67.830253,13.916496,15.737808,2.92732,2,<NA>,2,2,0.99812,15.459785
54,3,1,2009-01-01,NaN,2.274497e+06,7.626184e+03,NaN,NaN,NaN,NaN,...,67.830253,12.914442,16.739862,2.92732,2,<NA>,2,2,0.97293,16.432715
55,3,1,2010-01-01,NaN,4.481317e+06,4.298272e+04,NaN,NaN,NaN,NaN,...,67.830253,11.915127,17.739177,2.92732,2,<NA>,2,2,0.941299,17.374014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415169,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,2.587269,2.915811,-15.181727,NaN,454,<NA>,454,ETR,0.923801,24.360724
415170,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,2.587269,1.916496,-14.182411,NaN,454,<NA>,454,ETR,0.964819,25.325543
415173,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR,0.964819,25.325543
415176,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,1.752225,1.916496,-15.017456,NaN,454,<NA>,454,ETR,0.964819,25.325543


In [99]:
xwalk

,plant_id_eia,generator_id,emissions_unit_id_epa,subplant_id,pf_subplant_id,prime_mover,fuel_group,unit_id_pudl,capacity_xwalk,generator_operating_date,plant_id_epa,ppf,single_prime
0,1,WT2,WT2,0,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
1,1,WT1,WT1,1,0,WT,renew,<NA>,0.5,2011-10-01,1,1_WT_renew,True
2,1,5,5,2,1,IC,petroleum,<NA>,0.7,2000-12-01,1,1_IC_petroleum,True
3,1,3,3,3,1,IC,petroleum,<NA>,0.5,2010-12-01,1,1_IC_petroleum,True
4,1,2,2,4,1,IC,petroleum,<NA>,0.9,2000-12-01,1,1_IC_petroleum,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43015,65815,MPRES,MPRES,0,0,PV,renew,<NA>,1.0,2021-12-01,65815,65815_PV_renew,True
43016,65817,CLEA1,CLEA1,0,0,PV,renew,<NA>,3.6,2020-09-01,65817,65817_PV_renew,True
43017,65824,VSPRC,VSPRC,0,0,PV,renew,<NA>,1.1,2020-04-01,65824,65824_PV_renew,True
43018,65836,TOYAH,TOYAH,0,0,BA,other,<NA>,10.4,2021-10-01,65836,65836_BA_other,True


In [150]:
bins = [0,10,20,30,40,50,60,70,100]

labels = [1,2,3,4,5,6,7,8]

missing_1 = missing.assign(age_in_current_year = lambda x: np.where(x['age_in_current_year'].isnull(),30,x['age_in_current_year']),
essentials = lambda x: x['year'].astype(str) + "_" + x['prime_mover'] + "_" + x['fuel_group'],
ba_plus_essentials = lambda x: x['essentials'] + "_" + x['final_ba_code'],
age_range = lambda x: pd.cut(x['age_in_current_year'],bins=bins,labels=labels),
ba_plus_age = lambda x: x['ba_plus_essentials'] + "_" + x['age_range'])


In [151]:
missing_1

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range
0,3,3,2016,missing_years,ST,natural_gas,2,30.000000,2016_ST_natural_gas,2016_ST_natural_gas_2,3
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,30.000000,2008_CC_natural_gas,2008_CC_natural_gas_2,3
28,8,10,2019,missing_years,ST,coal,2,30.000000,2019_ST_coal,2019_ST_coal_2,3
46,8,6,2016,missing_years,ST,coal,2,30.000000,2016_ST_coal,2016_ST_coal_2,3
60,8,7,2016,missing_years,ST,coal,2,30.000000,2016_ST_coal,2016_ST_coal_2,3
...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2


In [114]:

hist_data_1 = (hist_data.
merge(xwalk[['plant_id_eia','generator_id','prime_mover','fuel_group']],
on=['plant_id_eia','generator_id','prime_mover'],how='left')
.assign(year = lambda x: x['report_date'].dt.year,
essentials = lambda x: x['year'].astype(str)  + "_" + x['prime_mover'] + "_" + x['fuel_group'],
ba_plus_essentials = lambda x: x['essentials'] + "_" + x['final_ba_code']
age_range=)
)

In [117]:
missing_1['essentials_present'] = np.where(missing_1['essentials'].isin(hist_data_1['essentials']),1,0)
missing_1['ba_plus_present'] = np.where(missing_1['ba_plus_essentials'].isin(hist_data_1['ba_plus_essentials']),1,0)
missing_1['ag']
missing_1

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,essentials_present,ba_plus_present
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,1,1
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,1,1
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,1,1
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,1,1
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,1,1
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,1,1
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,1,1
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,1,1


In [120]:
missing_1.query('ba_plus_present == 0',engine='python')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,essentials_present,ba_plus_present
110,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,1,0
5241,988,1,2016,missing_years,ST,petroleum,AEP,NaN,2016_ST_petroleum,2016_ST_petroleum_AEP,1,0
6102,1105,1,2017,missing_years,IC,petroleum,MISO,NaN,2017_IC_petroleum,2017_IC_petroleum_MISO,1,0
6118,1105,2,2017,missing_years,IC,petroleum,MISO,NaN,2017_IC_petroleum,2017_IC_petroleum_MISO,1,0
6984,1363,7A,2006,missing_years,CC,natural_gas,LGEE,NaN,2006_CC_natural_gas,2006_CC_natural_gas_LGEE,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
139246,60340,1,2015,missing_years,ST,biofuel,57,NaN,2015_ST_biofuel,2015_ST_biofuel_57,1,0
2242,6043,8,2009,fuel_switch,CC,renew,FPL,16.501027,2009_CC_renew,2009_CC_renew_FPL,0,0
2243,6043,8,2009,fuel_switch,CC,renew,FPL,16.501027,2009_CC_renew,2009_CC_renew_FPL,0,0
2244,6043,8,2009,fuel_switch,CC,renew,FPL,16.501027,2009_CC_renew,2009_CC_renew_FPL,0,0


,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR
133751,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR
133752,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR


In [145]:
missing_1.

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,essentials_present,ba_plus_present
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,1,1
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,1,1
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,1,1
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,1,1
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,1,1
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,1,1
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,1,1
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,1,1


In [144]:
hist_data_1

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR
133751,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR
133752,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,<NA>,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR


In [138]:
missing_1.merge(hist_data_1,on=['essentials'],how='left',indicator=True)

,plant_id_eia_x,generator_id_x,year_x,fuss,prime_mover_x,fuel_group_x,final_ba_code_x,age_in_current_year_x,essentials,ba_plus_essentials_x,...,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code_y,wage_scale,age_of_observation_secular_adj,fuel_group_y,year_y,ba_plus_essentials_y,_merge
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,...,2,<NA>,2,2,1.011758,23.20239,natural_gas,2016.0,2016_ST_natural_gas_2,both
1,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,...,2,<NA>,2,2,1.011758,23.20239,natural_gas,2016.0,2016_ST_natural_gas_2,both
2,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,...,2,<NA>,2,2,1.011758,23.20239,natural_gas,2016.0,2016_ST_natural_gas_2,both
3,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,...,2,<NA>,2,2,1.011758,23.20239,natural_gas,2016.0,2016_ST_natural_gas_2,both
4,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,...,2,<NA>,2,2,1.011758,23.20239,natural_gas,2016.0,2016_ST_natural_gas_2,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17322880,55505,BR2,2020,fuel_switch,GT,petroleum,529,19.586585,2020_GT_petroleum,2020_GT_petroleum_529,...,49,<NA>,49,EPE,1.013692,27.335436,petroleum,2020.0,2020_GT_petroleum_EPE,both
17322881,55505,BR2,2020,fuel_switch,GT,petroleum,529,19.586585,2020_GT_petroleum,2020_GT_petroleum_529,...,49,<NA>,49,EPE,1.013692,27.335436,petroleum,2020.0,2020_GT_petroleum_EPE,both
17322882,55505,BR2,2020,fuel_switch,GT,petroleum,529,19.586585,2020_GT_petroleum,2020_GT_petroleum_529,...,49,<NA>,49,EPE,1.013692,27.335436,petroleum,2020.0,2020_GT_petroleum_EPE,both
17322883,55505,BR2,2020,fuel_switch,GT,petroleum,529,19.586585,2020_GT_petroleum,2020_GT_petroleum_529,...,17,<NA>,17,DUKE,0.922797,26.923455,petroleum,2020.0,2020_GT_petroleum_DUKE,both


In [141]:
hist_data_1.query('plant_id_eia ==3 & generator_id == "3"',engine='python')

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials
60,3,3,2006-01-01,NaN,1.792376e+07,52430.611664,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2
61,3,3,2006-01-01,NaN,1.792376e+07,52430.611664,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2
62,3,3,2007-01-01,NaN,1.742333e+07,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2
63,3,3,2007-01-01,NaN,1.742333e+07,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2
64,3,3,2008-01-01,NaN,1.452576e+07,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2
65,3,3,2008-01-01,NaN,1.452576e+07,NaN,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.99812,15.459785,natural_gas,2008,2008_ST_natural_gas,2008_ST_natural_gas_2
66,3,3,2009-01-01,NaN,1.290395e+07,162040.790908,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.97293,16.432715,coal,2009,2009_ST_coal,2009_ST_coal_2
67,3,3,2009-01-01,NaN,1.290395e+07,162040.790908,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.97293,16.432715,natural_gas,2009,2009_ST_natural_gas,2009_ST_natural_gas_2
68,3,3,2010-01-01,NaN,9.347638e+06,222547.907467,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.941299,17.374014,coal,2010,2010_ST_coal,2010_ST_coal_2
69,3,3,2010-01-01,NaN,9.347638e+06,222547.907467,NaN,NaN,NaN,NaN,...,2,<NA>,2,2,0.941299,17.374014,natural_gas,2010,2010_ST_natural_gas,2010_ST_natural_gas_2


In [135]:

assign(age_in_current_year = lambda x: np.where(x['age_in_current_year'].isnull(),30,x['age_in_current_year']),
close_in_age = lambda x: np.where(abs(x['age_in_current_year'] - hist_data_1['age_in_current_year']) < 10,1,0)
)


ValueError: Length of values (134997) does not match length of index (14570)

In [29]:
missing_data_w_score = self.fill_in_ep_data()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

TypeError: 'DataFrame' object is not callable

In [5]:
missing_data_w_score.dtypes

plant_id_eia                     Int64
generator_id                    object
year                             Int64
fuss                            object
prime_mover                     string
fuel_group                      object
final_ba_code                   object
age_in_current_year            float64
essentials                      string
ba_plus_essentials              string
age_range                     category
ba_plus_age                     string
essentials_present               int64
ba_plus_age_present              int64
ba_plus_essentials_present       int64
fill_in_score                    int64
dtype: object

In [174]:
missing_data_w_score['test'] = missing_data_w_score['essentials_present'] + missing_data_w_score['ba_plus_present']

In [6]:
missing_data_w_score

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_age_present,ba_plus_essentials_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,0,1,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,0,1,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,0,1,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [10]:
missing_data_w_score.query('fill_in_score == 3',engine='python')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_age_present,ba_plus_essentials_present,fill_in_score
0,7527,2,2012,fuel_switch,CC,natural_gas,CAISO,26.168378,2012_CC_natural_gas,2012_CC_natural_gas_CAISO,3,2012_CC_natural_gas_CAISO_3,1,1,1,3
1,10333,GEN1,2013,fuel_switch,ST,coal,FPC,33.500342,2013_ST_coal,2013_ST_coal_FPC,4,2013_ST_coal_FPC_4,1,1,1,3
2,10333,GEN1,2014,fuel_switch,ST,coal,FPC,33.500342,2014_ST_coal,2014_ST_coal_FPC,4,2014_ST_coal_FPC_4,1,1,1,3
3,10333,GEN1,2015,fuel_switch,ST,coal,FPC,33.500342,2015_ST_coal,2015_ST_coal_FPC,4,2015_ST_coal_FPC_4,1,1,1,3
4,10333,GEN1,2016,fuel_switch,ST,coal,FPC,33.500342,2016_ST_coal,2016_ST_coal_FPC,4,2016_ST_coal_FPC_4,1,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [16]:
#from gencost.waterfall import create_fill_in_ep_thresholds

xwalk = self.xwalk

hist_data = (
    self.get_exa_by_generator()
    .merge(
        xwalk[["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]],
        on=["plant_id_eia", "generator_id", "prime_mover"],
        how="left",
    )
    .assign(year=lambda x: x["report_date"].dt.year)
    .pipe(self.create_fill_in_ep_thresholds)
)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [18]:
missing_data_w_score

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_age_present,ba_plus_essentials_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,0,1,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,0,1,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,0,1,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [17]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials,age_range,ba_plus_age
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2,7,2006_ST_coal_2_7
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2,7,2007_ST_coal_2_7
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2,7,2008_ST_coal_2_7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR,1,2019_CC_natural_gas_ETR_1
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133751,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133752,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1


In [20]:
missing_data_w_score.query('fill_in_score ==2')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_age_present,ba_plus_essentials_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,0,1,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,0,1,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,0,1,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2346,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,0,1,2
2347,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,0,1,2
2348,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,0,1,2
2349,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,0,1,2


In [21]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials,age_range,ba_plus_age
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2,7,2006_ST_coal_2_7
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2,7,2007_ST_coal_2_7
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2,7,2008_ST_coal_2_7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR,1,2019_CC_natural_gas_ETR_1
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133751,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133752,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1


In [27]:
hist_data.query('final_ba_code.isnull()')

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials,age_range,ba_plus_age
3640,310,2,2006-01-01,NaN,NaN,2.043756e+06,NaN,NaN,NaN,640.089648,...,<NA>,None,1.218299,15.045379,natural_gas,2006,2006_ST_natural_gas,<NA>,6,<NA>
3641,310,2,2006-01-01,NaN,NaN,2.043756e+06,NaN,NaN,NaN,640.089648,...,<NA>,None,1.218299,15.045379,petroleum,2006,2006_ST_petroleum,<NA>,6,<NA>
3642,310,2,2007-01-01,NaN,NaN,1.481319e+06,NaN,NaN,NaN,1149.303824,...,<NA>,None,1.205149,16.250528,natural_gas,2007,2007_ST_natural_gas,<NA>,6,<NA>
3643,310,2,2007-01-01,NaN,NaN,1.481319e+06,NaN,NaN,NaN,1149.303824,...,<NA>,None,1.205149,16.250528,petroleum,2007,2007_ST_petroleum,<NA>,6,<NA>
3644,310,2,2008-01-01,NaN,NaN,2.234114e+06,NaN,NaN,NaN,0.000000,...,<NA>,None,1.172143,17.422671,natural_gas,2008,2008_ST_natural_gas,<NA>,6,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133422,59073,5STB,2020-01-01,NaN,NaN,1.209106e+06,NaN,NaN,NaN,NaN,...,<NA>,NaN,1.151084,31.796804,natural_gas,2020,2020_ST_natural_gas,<NA>,1,<NA>
133423,59073,5STB,2020-01-01,NaN,NaN,1.209106e+06,NaN,NaN,NaN,NaN,...,<NA>,NaN,1.151084,31.796804,other,2020,2020_ST_other,<NA>,1,<NA>
133424,59073,5STB,2020-01-01,NaN,NaN,1.209106e+06,NaN,NaN,NaN,NaN,...,<NA>,NaN,1.151084,31.796804,natural_gas,2020,2020_ST_natural_gas,<NA>,1,<NA>
133425,59073,5STB,2020-01-01,NaN,NaN,1.209106e+06,NaN,NaN,NaN,NaN,...,<NA>,NaN,1.151084,31.796804,other,2020,2020_ST_other,<NA>,1,<NA>


In [28]:
missing_data_w_score.query('final_ba_code.isnull()')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_age_present,ba_plus_essentials_present,fill_in_score
922,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,1,2
932,310,3,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,1,2
940,310,4,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,1,2
1961,603,15,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,1,2
1972,603,16,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140623,2864,1,2015,missing_years,ST,coal,NaN,NaN,2015_ST_coal,<NA>,NaN,<NA>,1,0,1,2
140626,2864,2,2015,missing_years,ST,coal,NaN,NaN,2015_ST_coal,<NA>,NaN,<NA>,1,0,1,2
140667,7945,1,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,NaN,<NA>,1,0,1,2
140726,55858,UNT1,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,NaN,<NA>,1,0,1,2


In [22]:
(missing_data_w_score.query('fill_in_score ==2')
.merge(hist_data,on=['ba_plus_age'],how='left',indicator=True)
.query('_merge == "both"'))

,plant_id_eia_x,generator_id_x,year_x,fuss,prime_mover_x,fuel_group_x,final_ba_code_x,age_in_current_year_x,essentials_x,ba_plus_essentials_x,...,final_respondent_id,final_ba_code_y,wage_scale,age_of_observation_secular_adj,fuel_group_y,year_y,essentials_y,ba_plus_essentials_y,age_range_y,_merge
70,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,...,543,WALC,1.054843,14.07461,NaN,2006.0,<NA>,<NA>,6,both
71,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,...,543,WALC,1.081905,15.156515,NaN,2007.0,<NA>,<NA>,6,both
72,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,...,543,WALC,1.076,16.232515,NaN,2008.0,<NA>,<NA>,6,both
73,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,...,543,WALC,1.106098,17.338612,NaN,2009.0,<NA>,<NA>,6,both
74,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,...,543,WALC,1.069515,18.408127,NaN,2010.0,<NA>,<NA>,6,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370303,55858,UNT2,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,...,<NA>,NaN,1.151084,31.796804,natural_gas,2020.0,2020_ST_natural_gas,<NA>,1,both
370304,55858,UNT2,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,...,<NA>,NaN,1.151084,31.796804,other,2020.0,2020_ST_other,<NA>,1,both
370305,55858,UNT2,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,...,<NA>,NaN,1.151084,31.796804,natural_gas,2020.0,2020_ST_natural_gas,<NA>,1,both
370306,55858,UNT2,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,...,<NA>,NaN,1.151084,31.796804,other,2020.0,2020_ST_other,<NA>,1,both


In [30]:
xwalk = self.xwalk

hist_data = (
    self.get_exa_by_generator()
    .merge(
        xwalk[["plant_id_eia", "generator_id", "prime_mover", "fuel_group"]],
        on=["plant_id_eia", "generator_id", "prime_mover"],
        how="left",
    )
    .assign(year=lambda x: x["report_date"].dt.year)
    .pipe(self.create_fill_in_ep_thresholds)
)

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [ ]:
 .assign(
            ba_plus_age_present=lambda x: np.where(
                (x["final_ba_code"].isnull())
                | (x["final_ba_code"].isna() | x(["age_in_current_year"])),
                0,
                x["ba_plus_age_present"],
            ),
            ba_plus_essentials_present=lambda x: np.where(
                (x["final_ba_code"].isnull()) | (x["final_ba_code"].isna()),
                0,
                x["ba_plus_essentials_present"],
            ),
            fill_in_score=lambda x: x["essentials_present"]
            + x["ba_plus_essentials_present"]
            + x["ba_plus_age_present"],
        )
    )

In [ ]:
        ba_plus_age_present=lambda x: np.where(
                x["ba_plus_age"].isin(hist_data["ba_plus_age"]),
                1,
                0,
            ),
        )# make an exception of when there is no ba code or age
)

In [35]:
missing_data.assign(
            ba_plus_essentials_present=lambda x: np.where(
                x["ba_plus_essentials"].isin(hist_data["ba_plus_essentials"]),
                1,
               
            )

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.

In [32]:
missing_data = (
        self.find_missing_data()
        .pipe(self.create_fill_in_ep_thresholds)
        .assign(
            essentials_present=lambda x: np.where(
                x["essentials"].isin(hist_data["essentials"]), 1, 0
            ),
            ba_plus_essentials_present=lambda x: np.where(
                x["ba_plus_essentials"].isin(hist_data["ba_plus_essentials"]),
                1,
                np.where(
                (x["final_ba_code"].isnull()) | (x["final_ba_code"].isna()),
                0,
                x["ba_plus_essentials_present"],
            ),0,
            )))
            
    

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [37]:

missing_data = (
    self.find_missing_data()
    .pipe(self.create_fill_in_ep_thresholds)
    .assign(
        essentials_present=lambda x: np.where(
            x["essentials"].isin(hist_data["essentials"]), 1, 0
        ),
        ba_plus_essentials_present=lambda x: np.where(
            x["ba_plus_essentials"].isin(hist_data["ba_plus_essentials"]),
            1,
            0,
        ),
        ba_plus_age_present=lambda x: np.where(
            x["ba_plus_age"].isin(hist_data["ba_plus_age"]),
            1,
            0,
        ),
    )  # make score 0's when na's in ba code or age
    .assign(
        ba_plus_age_present=lambda x: np.where(
            (x["final_ba_code"].isnull())
            | (x["final_ba_code"].isna())
            | (x["age_in_current_year"].isna()) | (x["age_in_current_year"].isnull()),
            0,
            x["ba_plus_age_present"],
        ),
        ba_plus_essentials_present=lambda x: np.where(
            (x["final_ba_code"].isnull()) | (x["final_ba_code"].isna()),
            0,
            x["ba_plus_essentials_present"],
        ),
        fill_in_score=lambda x: x["essentials_present"]
        + x["ba_plus_essentials_present"]
        + x["ba_plus_age_present"],
    )
)

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [50]:
missing_data.query('fill_in_score == 2')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2346,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,1,0,2
2347,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,1,0,2
2348,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,1,0,2
2349,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018,2011_ST_other_gas,2011_ST_other_gas_MISO,3,2011_ST_other_gas_MISO_3,1,1,0,2


In [51]:
missing_data = self.fill_in_ep_data()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [52]:
missing_data

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [53]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,...,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials,age_range,ba_plus_age
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2,7,2006_ST_coal_2_7
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,...,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2,7,2007_ST_coal_2_7
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,...,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2,7,2008_ST_coal_2_7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR,1,2019_CC_natural_gas_ETR_1
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133751,60927,1A,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1
133752,60927,1B,2020-01-01,NaN,NaN,1.202763e+07,NaN,NaN,NaN,NaN,...,454,ETR,0.964819,25.325543,natural_gas,2020,2020_CC_natural_gas,2020_CC_natural_gas_ETR,1,2020_CC_natural_gas_ETR_1


In [75]:
missing_data

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [57]:
pd.set_option('display.max_columns',None)

In [59]:
(missing_data.query('fill_in_score ==2')
.merge(hist_data,on=['ba_plus_age'],how='left',indicator=True)
.query('_merge == "both"'))

,plant_id_eia_x,generator_id_x,year_x,fuss,prime_mover_x,fuel_group_x,final_ba_code_x,age_in_current_year_x,essentials_x,ba_plus_essentials_x,age_range_x,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,plant_id_eia_y,generator_id_y,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,prime_mover_y,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year_y,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code_y,wage_scale,age_of_observation_secular_adj,fuel_group_y,year_y,essentials_y,ba_plus_essentials_y,age_range_y,_merge


In [76]:
(missing_data.query('fill_in_score ==2')
.merge(hist_data.drop(columns=['plant_id_eia','generator_id','year','prime_mover','fuel_group','final_ba_code','age_in_current_year','age_range','ba_plus_age','essentials']),on=['ba_plus_essentials'],how='left',indicator=True)
.query('_merge == "both"')
.drop_duplicates(subset=['plant_id_eia','generator_id','year']))

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.338707e+02,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
13,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2,2008-01-01,NaN,NaN,6.091393e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.582251e+05,NaN,NaN,NaN,NaN,NaN,NaN,886.0,886.0,4.820953e+06,NaN,NaN,4.820953e+06,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,170.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.668720,13.916496,-7.184464,2.927320,2,<NA>,2,0.99812,15.459785,both
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2,2019-01-01,NaN,NaN,4.135949e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.952539e+03,NaN,NaN,NaN,NaN,NaN,NaN,6.0,5.0,2.945400e+04,NaN,NaN,2.945400e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,64.914442,2.915811,26.738493,2.927320,2,<NA>,2,1.078991,26.305583,both
45,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.338707e+02,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
65,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.338707e+02,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
919359,57185,U006,2015,missing_years,CC,natural_gas,NYIS,NaN,2015_CC_natural_gas,2015_CC_natural_gas_NYIS,NaN,2015_CC_natural_gas_NYIS_nan,1,1,0,2,2015-01-01,NaN,NaN,1.110898e+07,NaN,NaN,NaN,136227.0,NaN,NaN,NaN,NaN,1.047181e+06,NaN,NaN,NaN,12720.854000,NaN,NaN,28.0,46.0,3.168856e+06,NaN,NaN,3.130468e+06,NaN,NaN,38388.250486,NaN,NaN,10023,NYIS,NY,170.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,11.085558,6.9158

In [67]:
(missing_data.query('fill_in_score ==3')
.merge(hist_data.drop(columns=['plant_id_eia','generator_id','year','prime_mover','fuel_group','final_ba_code','age_in_current_year','age_range','essentials','ba_plus_essentials']),on=['ba_plus_age'],how='left',indicator=True)
.query('_merge == "both"')
.drop_duplicates(subset=['plant_id_eia','generator_id','year']))

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,7527,2,2012,fuel_switch,CC,natural_gas,CAISO,26.168378,2012_CC_natural_gas,2012_CC_natural_gas_CAISO,3,2012_CC_natural_gas_CAISO_3,1,1,1,3,2012-01-01,NaN,NaN,3.715400e+06,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,265316.163925,NaN,NaN,NaN,0.000000,NaN,NaN,34.0,32.0,1205096.0,NaN,NaN,1.205096e+06,NaN,NaN,0.000000,NaN,NaN,9216,IID,CA,89.9,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,18.584531,9.916496,3.731347,5.205650,648,<NA>,648,1.23754,22.301356,both
18,10333,GEN1,2013,fuel_switch,ST,coal,FPC,33.500342,2013_ST_coal,2013_ST_coal_FPC,4,2013_ST_coal_FPC_4,1,1,1,3,2013-01-01,NaN,3.913836e+07,NaN,NaN,NaN,NaN,120324.961137,NaN,NaN,NaN,3.892536e+06,NaN,NaN,NaN,NaN,10918.037449,NaN,NaN,9.0,13.0,4128033.0,NaN,4.115381e+06,NaN,NaN,NaN,12652.116668,NaN,NaN,6455,FPC,FL,739.2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,28.251882,8.914442,-9.924067,67.961165,55,<NA>,55,0.940483,19.278976,both
21,10333,GEN1,2014,fuel_switch,ST,coal,FPC,33.500342,2014_ST_coal,2014_ST_coal_FPC,4,2014_ST_coal_FPC_4,1,1,1,3,2014-01-01,NaN,4.412765e+07,NaN,NaN,NaN,NaN,113056.925179,NaN,NaN,NaN,4.350971e+06,NaN,NaN,NaN,NaN,11840.169822,NaN,NaN,14.0,9.0,4870790.0,NaN,4.858343e+06,NaN,NaN,NaN,12447.283071,NaN,NaN,6455,FPC,FL,739.2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,29.251198,7.915127,-8.924752,67.961165,55,<NA>,55,0.960971,20.239947,both
24,10333,GEN1,2015,fuel_switch,ST,coal,FPC,33.500342,2015_ST_coal,2015_ST_coal_FPC,4,2015_ST_coal_FPC_4,1,1,1,3,2015-01-01,NaN,3.304029e+07,NaN,NaN,NaN,NaN,106924.117914,NaN,NaN,NaN,3.219024e+06,NaN,NaN,NaN,NaN,7774.117138,NaN,NaN,4.0,6.0,3551209.0,NaN,3.539754e+06,NaN,NaN,NaN,11455.257331,NaN,NaN,6455,FPC,FL,739.2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,30.250513,6.915811,-7.925436,67.961165,55,<NA>,55,0.97828,21.218227,both
27,10333,GEN1,2016,fuel_switch,ST,coal,FPC,33.500342,2016_ST_coal,2016_ST_coal_FPC,4,2016_ST_coal_FPC_4,1,1,1,3,2016-01-01,NaN,3.895418e+07,NaN,NaN,NaN,NaN,120981.265059,NaN,NaN,NaN,3.779808e+06,NaN,NaN,NaN,NaN,15260.807330,NaN,NaN,4.0,4.0,4172327.0,NaN,4.159409e+06,NaN,NaN,NaN,12918.011673,NaN,NaN,6455,FPC,FL,739.2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,31.249829,5.916496,-6.926121,67.961165,55,<NA>,55,1.013763,22.231991,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100680,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3,2016-01-01,NaN,NaN,5.933650e+04,NaN,NaN,NaN,3232.250000,NaN,NaN,NaN,NaN,5984.599500,NaN,NaN,NaN,325.900500,NaN,NaN,25.0,28.0,112

In [69]:
(missing_data.query('fill_in_score ==1')
.merge(hist_data.drop(columns=['plant_id_eia','generator_id','year','prime_mover','fuel_group','final_ba_code','age_in_current_year','age_range','ba_plus_age','ba_plus_essentials']),on=['essentials'],how='left',indicator=True)
.query('_merge == "both"')
.drop_duplicates(subset=['plant_id_eia','generator_id','year']))

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,NaN,2006_GT_natural_gas_EPE_nan,1,0,0,1,2006-01-01,NaN,NaN,261284.189244,NaN,NaN,NaN,3703.0,NaN,NaN,NaN,NaN,27853.222222,NaN,NaN,NaN,240.222222,NaN,NaN,85.0,88.0,25411.0,NaN,NaN,25055.900068,NaN,NaN,355.099932,NaN,NaN,195,SOCO,AL,80.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,9.670089,15.915127,-12.342141,107.300831,2,<NA>,2,1.050357,13.40423,both
1184,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
1990,310,3,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
2840,310,4,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
3690,603,15,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
778776,2864,1,2015,missing_years,ST,coal,NaN,NaN,2015_ST_coal,<NA>,NaN,<NA>,1,0,0,1,2015-01-01,NaN,NaN,59476.795354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,51.994385,NaN,NaN,NaN,NaN,NaN,NaN,6.0,6.0,4936.0,NaN,NaN,4936.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,60.914442,6.915811,22.738493,2.927320,2,<NA>,2,0.973045,22.190632,both
779796,2864,2,2015,miss

In [79]:
zero = missing_data.query('fill_in_score == 0')

In [84]:
zero.shape[0]

39

In [86]:
hist_data

,plant_id_eia,generator_id,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code,wage_scale,age_of_observation_secular_adj,fuel_group,year,essentials,ba_plus_essentials,age_range,ba_plus_age
0,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4.120480e+03,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1123980.0,NaN,1.120702e+06,3.278278e+03,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,ST,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,67.830253,15.915127,13.739177,2.92732,2,<NA>,2,2,1.050357,13.40423,coal,2006,2006_ST_coal,2006_ST_coal_2,7,2006_ST_coal_2_7
1,3,1,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4.120480e+03,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1123980.0,NaN,1.120702e+06,3.278278e+03,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,ST,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,67.830253,15.915127,13.739177,2.92732,2,<NA>,2,2,1.050357,13.40423,natural_gas,2006,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7
2,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,954656.813326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,1028301.0,NaN,1.028301e+06,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,ST,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.914442,67.830253,14.915811,14.738493,2.92732,2,<NA>,2,2,1.057435,14.461664,coal,2007,2007_ST_coal,2007_ST_coal_2,7,2007_ST_coal_2_7
3,3,1,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,954656.813326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,1028301.0,NaN,1.028301e+06,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,ST,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.914442,67.830253,14.915811,14.738493,2.92732,2,<NA>,2,2,1.057435,14.461664,natural_gas,2007,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7
4,3,1,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,865288.470536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,15.0,928095.0,NaN,9.280950e+05,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,ST,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,53.913758,67.830253,13.916496,15.737808,2.92732,2,<NA>,2,2,0.99812,15.459785,coal,2008,2008_ST_coal,2008_ST_coal_2,7,2008_ST_coal_2_7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133749,60926,1C,2019-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.449205e+06,NaN,NaN,NaN,NaN,NaN,NaN,35.0,35.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,11241,MISO,LA,500.0,CC,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-0.328542,2.587269,2.915811,-15.181727,NaN,454,<NA>,454,ETR,0.923801,24.360724,natural_gas,2019,2019_CC_natural_gas,2019_CC_natural_gas_ETR,1,2019_CC_natural_gas_ETR_1
133750,60926,1C,2020-01-01,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.443787e+06,NaN,NaN,NaN,NaN,NaN,NaN,36.0,37.0,0.0,NaN,NaN,0.000000e+00,NaN,NaN,NaN,NaN,NaN,11241,MISO,LA,500.0,CC,1.0,1.0,1.0,1.0,1.0,1.0

In [73]:
test = self.fill_in_ep_data()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [74]:
test

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [87]:
df = self.get_gf923_by_generator(counterfactuals=True)

In [91]:
df.query('net_mwh == 0 & mmbtu == 0 & report_date >= "2006-"')

,report_date,plant_id_eia,generator_id,prime_mover_code,energy_source_code,energy_source_code_num,net_mwh,mmbtu,fuel_consumed_for_electricity_mmbtu,fuel_group
1226367,2001-01-01,9,1,GT,DFO,energy_source_code_2,0.0,0.0,0.0,petroleum
1247942,2001-01-01,47,GT1,GT,NG,energy_source_code_1,0.0,0.0,0.0,natural_gas
1247943,2001-01-01,47,GT2,GT,NG,energy_source_code_1,0.0,0.0,0.0,natural_gas
1247944,2001-01-01,47,GT3,GT,NG,energy_source_code_1,0.0,0.0,0.0,natural_gas
1247945,2001-01-01,47,GT4,GT,NG,energy_source_code_1,0.0,0.0,0.0,natural_gas
...,...,...,...,...,...,...,...,...,...,...
6807129,2021-12-01,65650,15447,BA,MWH,energy_source_code_1,0.0,0.0,0.0,other
6807153,2021-12-01,65652,NBRIA,PV,SUN,energy_source_code_1,0.0,0.0,0.0,renew
6807225,2021-12-01,65659,PV1,PV,SUN,energy_source_code_1,0.0,0.0,0.0,renew
6807261,2021-12-01,65667,EWS1,PV,SUN,energy_source_code_1,0.0,0.0,0.0,renew


In [100]:
k

,plant_id_eia,generator_id,report_date,net_mwh,mmbtu
1,1,1,2020-01-01,0.0,0.0
5,1,2,2020-01-01,0.0,0.0
9,1,3,2020-01-01,0.0,0.0
13,1,5,2020-01-01,0.0,0.0
191,3,A1ST,2006-01-01,0.0,0.0
...,...,...,...,...,...
431169,64816,GEN1,2020-01-01,0.0,0.0
431172,64816,GEN2,2020-01-01,0.0,0.0
431175,64816,GEN3,2020-01-01,0.0,0.0
431212,64836,CATAL,2020-01-01,0.0,0.0


In [103]:
df_860 = self.get_860_by_x(subplant_id_col='generator_id')

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(


In [104]:
df_860

,utility_id_eia,balancing_authority_code_eia,state,plant_id_eia,generator_id,report_date,capacity_mw,prime_mover,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_in_current_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,final_ba_code
0,195,SOCO,AL,2,1,2001-01-01,45.0,HY,1,1,1,1,1,1,1,1,1,1,1,1,37.505818,58.420260,20.914442,-22.507147,2.92732,2,<NA>,2,2
1,195,SOCO,AL,3,1,2001-01-01,153.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,46.915811,67.830253,20.914442,8.739862,2.92732,2,<NA>,2,2
2,195,SOCO,AL,3,2,2001-01-01,153.1,ST,1,1,1,1,1,1,1,1,1,1,1,1,46.505133,67.419576,20.914442,8.329184,2.92732,2,<NA>,2,2
3,195,SOCO,AL,3,3,2001-01-01,272.0,ST,1,1,1,1,1,1,1,1,1,1,1,1,41.505818,62.420260,20.914442,3.329869,NaN,2,<NA>,2,2
4,195,SOCO,AL,3,4,2001-01-01,403.7,ST,1,1,1,1,1,1,1,1,1,1,1,1,31.085558,52.000000,20.914442,-7.090392,2.92732,2,<NA>,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427134,61980,NaN,CA,65824,VSPRC,2022-01-01,1.1,PV,1,1,1,1,1,1,1,1,1,1,1,1,1.752225,1.667351,-0.084873,-1.315511,NaN,<NA>,<NA>,<NA>,NaN
427135,65076,ERCO,TX,65831,LNSTR,2022-01-01,9.9,BA,1,1,1,1,1,1,1,1,1,1,1,1,NaN,NaN,-0.084873,NaN,NaN,<NA>,<NA>,<NA>,ERCO
427136,65076,ERCO,TX,65836,TOYAH,2022-01-01,10.4,BA,1,1,1,1,1,1,1,1,1,1,1,1,0.251882,0.167009,-0.084873,-1.996059,NaN,<NA>,<NA>,<NA>,ERCO
427137,65076,ERCO,TX,65838,HOEFS,2022-01-01,2.0,BA,1,1,1,1,1,1,1,1,1,1,1,1,0.670773,0.585900,-0.084873,-1.577168,NaN,<NA>,<NA>,<NA>,ERCO


In [158]:
missing_data['year'].dt.strftime('%Y-%m-%d')

AttributeError: Can only use .dt accessor with datetimelike values

In [161]:
m['year'].dt.strftime('%Y-%m-%d')

AttributeError: Can only use .dt accessor with datetimelike values

In [159]:
pd.to_datetime(missing_data['year'],format='%Y-%m-%d')

#.dt.strftime('%Y%m%d')

0      1970-01-01 00:00:00.000002016
14     1970-01-01 00:00:00.000002008
28     1970-01-01 00:00:00.000002019
46     1970-01-01 00:00:00.000002016
60     1970-01-01 00:00:00.000002016
                    ...             
2392   1970-01-01 00:00:00.000002016
2393   1970-01-01 00:00:00.000002017
2394   1970-01-01 00:00:00.000002018
2395   1970-01-01 00:00:00.000002019
2396   1970-01-01 00:00:00.000002020
Name: year, Length: 14570, dtype: datetime64[ns]

In [152]:
missing_data

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2
14,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2
46,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
60,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2392,55505,BR2,2016,fuel_switch,GT,petroleum,529,19.586585,2016_GT_petroleum,2016_GT_petroleum_529,2,2016_GT_petroleum_529_2,1,1,1,3
2393,55505,BR2,2017,fuel_switch,GT,petroleum,529,19.586585,2017_GT_petroleum,2017_GT_petroleum_529,2,2017_GT_petroleum_529_2,1,1,1,3
2394,55505,BR2,2018,fuel_switch,GT,petroleum,529,19.586585,2018_GT_petroleum,2018_GT_petroleum_529,2,2018_GT_petroleum_529_2,1,1,1,3
2395,55505,BR2,2019,fuel_switch,GT,petroleum,529,19.586585,2019_GT_petroleum,2019_GT_petroleum_529,2,2019_GT_petroleum_529_2,1,1,1,3


In [114]:
k = self.find_missing_data()

/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:199: FutureWarning: Support for multi-dimensional indexing (e.g. `obj[:, None]`) is deprecated and will be removed in a future version.  Convert to a numpy array before indexing instead.
  np.repeat(df[old_cols].sum(axis=1)[:, np

In [117]:
k.query('fuss == "zeroes"')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,1,1,2020,zeroes,IC,petroleum,Alaska,20.999316
1,1,2,2020,zeroes,IC,petroleum,Alaska,20.999316
2,1,3,2020,zeroes,IC,petroleum,Alaska,11.000684
3,1,5,2020,zeroes,IC,petroleum,Alaska,20.999316
4,3,A1ST,2006,zeroes,CC,natural_gas,2,21.585216
...,...,...,...,...,...,...,...,...
42123,64816,GEN1,2020,zeroes,IC,natural_gas,ERCO,1.248460
42124,64816,GEN2,2020,zeroes,IC,natural_gas,ERCO,1.248460
42125,64816,GEN3,2020,zeroes,IC,natural_gas,ERCO,1.248460
42126,64836,CATAL,2020,zeroes,PV,renew,CAISO,0.999316


In [166]:
m = self.fill_in_ep_data()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [165]:
pd.to_datetime(m['year'],format="%Y")

785664    2020-01-01
785667    2020-01-01
785670    2020-01-01
785673    2020-01-01
335705    2006-01-01
             ...    
3996801   2020-01-01
3996804   2020-01-01
3996807   2020-01-01
3996810   2020-01-01
3996813   2020-01-01
Name: year, Length: 45634, dtype: datetime64[ns]

In [169]:
missing_data['fill_in_match'] = np.where(missing_data['fill_in_score'] == 3,'essentials_ba_age_group',(np.where(missing_data['fill_in_score']==2,'essentials_ba','essentials')))

In [171]:
missing_data.query('fill_in_score == 1')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,fill_in_match
110,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,NaN,2006_GT_natural_gas_EPE_nan,1,0,0,1,essentials
922,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,essentials
932,310,3,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,essentials
940,310,4,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,essentials
1961,603,15,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,essentials
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140623,2864,1,2015,missing_years,ST,coal,NaN,NaN,2015_ST_coal,<NA>,NaN,<NA>,1,0,0,1,essentials
140626,2864,2,2015,missing_years,ST,coal,NaN,NaN,2015_ST_coal,<NA>,NaN,<NA>,1,0,0,1,essentials
140667,7945,1,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,NaN,<NA>,1,0,0,1,essentials
140726,55858,UNT1,2015,missing_years,GT,natural_gas,None,NaN,2015_GT_natural_gas,<NA>,NaN,<NA>,1,0,0,1,essentials


In [167]:
m

,plant_id_eia,generator_id,fuss,final_ba_code,age_in_current_year,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj
785664,1,1,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848
785667,1,2,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848
785670,1,3,zeroes,Alaska,11.000684,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848
785673,1,5,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848
335705,3,1,fuel_switch,2,67.830253,2006-01-01,NaN,1.008871e+07,29511.495021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4120.479810,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1123980.0,NaN,1.120702e+06,3278.277774,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,15.915127,13.739177,2.927320,2,<NA>,2,1.050357,13.40423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996801,64815,GEN2,zeroes,ERCO,2.420260,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436
3996804,64815,GEN3,zeroes,ERCO,2.420260,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436
3996807,64816,GEN1,zeroes,ERCO,1.248460,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436
3996810,64816,GEN2,zeroes,ERCO,1.248460,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,N

In [123]:
m.query('fill_in_score == 1')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,NaN,2006_GT_natural_gas_EPE_nan,1,0,0,1,2006-01-01,NaN,NaN,261284.189244,NaN,NaN,NaN,3703.0,NaN,NaN,NaN,NaN,27853.222222,NaN,NaN,NaN,240.222222,NaN,NaN,85.0,88.0,25411.0,NaN,NaN,25055.900068,NaN,NaN,355.099932,NaN,NaN,195,SOCO,AL,80.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,9.670089,15.915127,-12.342141,107.300831,2,<NA>,2,1.050357,13.40423,both
1184,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
1990,310,3,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
2840,310,4,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
3690,603,15,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996801,64815,GEN2,2020,zeroes,IC,natural_gas,ERCO,2.42026,2020_IC_natural_gas,2020_IC_natural_gas_ERCO,1,2020_IC_natural_gas_ERCO_1,1,0,0,1,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.2806

In [124]:
m.query('fill_in_score == 2')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,3,3,2016,missing_years,ST,natural_gas,2,NaN,2016_ST_natural_gas,2016_ST_natural_gas_2,NaN,2016_ST_natural_gas_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
13,3,A1CT2,2008,missing_years,CC,natural_gas,2,NaN,2008_CC_natural_gas,2008_CC_natural_gas_2,NaN,2008_CC_natural_gas_2_nan,1,1,0,2,2008-01-01,NaN,NaN,6.091393e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,958225.064706,NaN,NaN,NaN,NaN,NaN,NaN,886.0,886.0,4.820953e+06,NaN,NaN,4.820953e+06,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,170.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.668720,13.916496,-7.184464,2.927320,2,<NA>,2,0.99812,15.459785,both
28,8,10,2019,missing_years,ST,coal,2,NaN,2019_ST_coal,2019_ST_coal_2,NaN,2019_ST_coal_2_nan,1,1,0,2,2019-01-01,NaN,NaN,4.135949e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1952.538969,NaN,NaN,NaN,NaN,NaN,NaN,6.0,5.0,2.945400e+04,NaN,NaN,2.945400e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,64.914442,2.915811,26.738493,2.927320,2,<NA>,2,1.078991,26.305583,both
45,8,6,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
65,8,7,2016,missing_years,ST,coal,2,NaN,2016_ST_coal,2016_ST_coal_2,NaN,2016_ST_coal_2_nan,1,1,0,2,2016-01-01,NaN,NaN,1.618925e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,1.189000e+04,NaN,NaN,1.189000e+04,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967277,63919,1,2019,zeroes,ST,natural_gas,MISO,3.167693,2019_ST_natural_gas,2019_ST_natural_gas_MISO,1,2019_ST_natural_gas_MISO_1,1,1,0,2,2019-01-01,NaN,8.779109e+06,8.995000e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,787574.221749,8160.783309,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,8.711970e+05,NaN,862361.320108,8.835680e+03,NaN,NaN,NaN,NaN,NaN,5748,GLHB,IL,183.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,65.418207,2.915811,27.24225

In [125]:
m.query('fill_in_score == 1')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
0,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,NaN,2006_GT_natural_gas_EPE_nan,1,0,0,1,2006-01-01,NaN,NaN,261284.189244,NaN,NaN,NaN,3703.0,NaN,NaN,NaN,NaN,27853.222222,NaN,NaN,NaN,240.222222,NaN,NaN,85.0,88.0,25411.0,NaN,NaN,25055.900068,NaN,NaN,355.099932,NaN,NaN,195,SOCO,AL,80.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,9.670089,15.915127,-12.342141,107.300831,2,<NA>,2,1.050357,13.40423,both
1184,310,2,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
1990,310,3,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
2840,310,4,2016,missing_years,ST,natural_gas,None,NaN,2016_ST_natural_gas,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,NaN,161892.503123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,233.870694,NaN,NaN,NaN,NaN,NaN,NaN,10.0,10.0,11890.0,NaN,NaN,11890.000000,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,61.913758,5.916496,23.737808,2.927320,2,<NA>,2,1.011758,23.20239,both
3690,603,15,2016,missing_years,ST,petroleum,None,NaN,2016_ST_petroleum,<NA>,NaN,<NA>,1,0,0,1,2016-01-01,NaN,373995.386529,771038.177507,NaN,NaN,NaN,0.0,NaN,NaN,NaN,75819.390226,17604.609865,NaN,NaN,NaN,0.000000,NaN,NaN,9.0,9.0,109697.0,NaN,35829.667535,73867.332465,NaN,NaN,0.000000,NaN,NaN,195,SOCO,AL,272.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.668720,5.916496,17.492771,0.582288,2,<NA>,2,1.011758,23.20239,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996801,64815,GEN2,2020,zeroes,IC,natural_gas,ERCO,2.42026,2020_IC_natural_gas,2020_IC_natural_gas_ERCO,1,2020_IC_natural_gas_ERCO_1,1,0,0,1,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.2806

In [128]:
m = m.sort_values(by=['plant_id_eia','generator_id','year'],ascending=True)

In [133]:
m.query('plant_id_eia == 1',engine='python')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
785664,1,1,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785667,1,2,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785670,1,3,2020,zeroes,IC,petroleum,Alaska,11.000684,2020_IC_petroleum,2020_IC_petroleum_Alaska,2,2020_IC_petroleum_Alaska_2,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785673,1,5,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both


In [131]:
m.query('fill_in_score == 1')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
785664,1,1,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785667,1,2,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785670,1,3,2020,zeroes,IC,petroleum,Alaska,11.000684,2020_IC_petroleum,2020_IC_petroleum_Alaska,2,2020_IC_petroleum_Alaska_2,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785673,1,5,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
0,9,1,2006,missing_years,GT,natural_gas,EPE,NaN,2006_GT_natural_gas,2006_GT_natural_gas_EPE,NaN,2006_GT_natural_gas_EPE_nan,1,0,0,1,2006-01-01,NaN,NaN,261284.189244,NaN,NaN,NaN,3703.0,NaN,NaN,NaN,NaN,27853.222222,NaN,NaN,NaN,240.222222,NaN,NaN,85.0,88.0,25411.0,NaN,NaN,25055.900068,NaN,NaN,355.099932,NaN,NaN,195,SOCO,AL,80.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,9.670089,15.915127,-12.342141,107.300831,2,<NA>,2,1.050357,13.40423,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996801,64815,GEN2,2020,zeroes,IC,natural_gas,ERCO,2.420260,2020_IC_natural_gas,2020_IC_natural_gas_ERCO,1,2020_IC_natural_gas_ERCO_1,1,0,0,1,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,

In [132]:
m.query('fill_in_score == 3')

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
335705,3,1,2006,fuel_switch,ST,natural_gas,2,67.830253,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7,1,1,1,3,2006-01-01,NaN,1.008871e+07,29511.495021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4120.479810,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1123980.0,NaN,1.120702e+06,3278.277774,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,15.915127,13.739177,2.927320,2,<NA>,2,1.050357,13.40423,both
335711,3,1,2007,fuel_switch,ST,natural_gas,2,67.830253,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7,1,1,1,3,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,954656.813326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,1028301.0,NaN,1.028301e+06,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.914442,14.915811,14.738493,2.927320,2,<NA>,2,1.057435,14.461664,both
335717,3,1,2008,fuel_switch,ST,natural_gas,2,67.830253,2008_ST_natural_gas,2008_ST_natural_gas_2,7,2008_ST_natural_gas_2_7,1,1,1,3,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,865288.470536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,15.0,928095.0,NaN,9.280950e+05,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,53.913758,13.916496,15.737808,2.927320,2,<NA>,2,0.99812,15.459785,both
335723,3,1,2009,fuel_switch,ST,natural_gas,2,67.830253,2009_ST_natural_gas,2009_ST_natural_gas_2,7,2009_ST_natural_gas_2_7,1,1,1,3,2009-01-01,NaN,2.274497e+06,7626.184443,NaN,NaN,NaN,NaN,NaN,NaN,NaN,216274.279645,5633.720355,NaN,NaN,NaN,NaN,NaN,NaN,44.0,44.0,243753.0,NaN,2.429384e+05,814.550756,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,54.915811,12.914442,16.739862,2.927320,2,<NA>,2,0.97293,16.432715,both
335729,3,1,2010,fuel_switch,ST,natural_gas,2,67.830253,2010_ST_natural_gas,2010_ST_natural_gas_2,7,2010_ST_natural_gas_2_7,1,1,1,3,2010-01-01,NaN,4.481317e+06,42982.723242,NaN,NaN,NaN,NaN,NaN,NaN,NaN,428843.516663,6490.483337,NaN,NaN,NaN,NaN,NaN,NaN,14.0,14.0,469201.0,NaN,4.647434e+05,4457.603912,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,55.915127,11.915127,17.739177,2.927320,2,<NA>,2,0.941299,17.374014,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
334978,63478,CT-1,2020,zeroes,GT,natural_gas,PJM,5.166324,2020_GT_natural_gas,2020_GT_natural_gas_PJM,1,2020_GT_natural_gas_PJM_1,1,1,1,3,2020-01-01,NaN,NaN,798932.675676,NaN,NaN,NaN,77674.302255,NaN,NaN,NaN,NaN,71619.989676,NaN,NaN,NaN,6988.971030,NaN,Na

In [ ]:
    .merge(
                df_860.assign(year=lambda x: x["report_date"].dt.year)[
                    [
                        "plant_id_eia",
                        "generator_id",
                        "year",
                        "final_ba_code",
                        "age_in_current_year",
                    ]
                ],
                on=["plant_id_eia", "generator_id", "year"],
                how="left",
                # indicator=True,
            )
        )[
            [
                "plant_id_eia",
                "generator_id",
                "year",
                "fuss",
                "prime_mover",
                "fuel_group",
                "final_ba_code",
                "age_in_current_year",
            ]
        ]

In [105]:
from gencost.constants import FOSSIL_PRIME_MOVER_MAP, FUEL_GROUP_MAP, GET_860_GEN_COLS

In [111]:
single_fuel_switch = (
            hist_data.pipe(self.filter_to_single_fuel_generators)
            .assign(
                n_fuels=lambda x: x.groupby(["plant_id_eia", "generator_id"])[
                    "mmbtu"
                ].transform("nunique")
            )
            .query("n_fuels > 1")
            .assign(
                fuss=lambda x: "fuel_switch",
                year=lambda x: x["report_date"].dt.year,
                fuel=lambda x: x.groupby(["plant_id_eia", "generator_id"])[
                    "mmbtu"
                ].transform("last"),
                fuel_reported_is_latest_fuel=lambda x: np.where(
                    x["mmbtu"] == x["fuel"], True, False
                ),
            )
            .query("fuel_reported_is_latest_fuel == False")
            .assign(fuel_group=lambda x: x["fuel"].str.replace("_mmbtu", ""))
            .merge(xwalk, on=["plant_id_eia", "generator_id", "fuel_group"], how="left")
        )

zero_reported = (
    self.get_gf923_by_generator(counterfactuals=True)
    .groupby(
        [
            "plant_id_eia",
            "generator_id",
            pd.Grouper(key="report_date", freq="YS"),
            'prime_mover_code','fuel_group'
        ]
    )
    .agg({"net_mwh": "sum", "mmbtu": "sum"})
    .reset_index()
    .query(
        'net_mwh == 0 & mmbtu == 0 & report_date >= "2006-01-01" & report_date <= "2020-01-01"'
    )
    .assign(
        prime_mover=lambda x: x.prime_mover_code.replace(FOSSIL_PRIME_MOVER_MAP),
        year=lambda x: x["report_date"].dt.year,
        fuss = 'zeroes'

    )
)



In [112]:
zero_and_fuel_switch = (pd.concat([zero_reported,single_fuel_switch])
    .merge(
                df_860.assign(year=lambda x: x["report_date"].dt.year)[
                    [
                        "plant_id_eia",
                        "generator_id",
                        "year",
                        "final_ba_code",
                        "age_in_current_year",
                    ]
                ],
                on=["plant_id_eia", "generator_id", "year"],
                how="left",
                # indicator=True,
            )
        )[
            [
                "plant_id_eia",
                "generator_id",
                "year",
                "fuss",
                "prime_mover",
                "fuel_group",
                "final_ba_code",
                "age_in_current_year",
            ]
        ]


In [113]:
zero_and_fuel_switch

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year
0,1,1,2020,zeroes,IC,petroleum,Alaska,20.999316
1,1,2,2020,zeroes,IC,petroleum,Alaska,20.999316
2,1,3,2020,zeroes,IC,petroleum,Alaska,11.000684
3,1,5,2020,zeroes,IC,petroleum,Alaska,20.999316
4,3,A1ST,2006,zeroes,CC,natural_gas,2,21.585216
...,...,...,...,...,...,...,...,...
42170,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018
42171,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018
42172,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018
42173,55088,ST1,2011,fuel_switch,ST,other_gas,MISO,20.334018


In [145]:
m.columns

Index(['plant_id_eia', 'generator_id', 'year', 'fuss', 'prime_mover',
       'fuel_group', 'final_ba_code', 'age_in_current_year', 'essentials',
       'ba_plus_essentials', 'age_range', 'ba_plus_age', 'essentials_present',
       'ba_plus_essentials_present', 'ba_plus_age_present', 'fill_in_score',
       'report_date', 'biofuel_mmbtu', 'coal_mmbtu', 'natural_gas_mmbtu',
       'nuclear_mmbtu', 'other_mmbtu', 'other_gas_mmbtu', 'petroleum_mmbtu',
       'petroleum_coke_mmbtu', 'renew_mmbtu', 'biofuel_net_mwh',
       'coal_net_mwh', 'natural_gas_net_mwh', 'nuclear_net_mwh',
       'other_net_mwh', 'other_gas_net_mwh', 'petroleum_net_mwh',
       'petroleum_coke_net_mwh', 'renew_net_mwh', 'generator_starts',
       'fuel_starts', 'gross_generation_mwh', 'biofuel_gross_mwh',
       'coal_gross_mwh', 'natural_gas_gross_mwh', 'other_gross_mwh',
       'other_gas_gross_mwh', 'petroleum_gross_mwh',
       'petroleum_coke_gross_mwh', 'renew_gross_mwh', 'utility_id_eia',
       'balancing_aut

In [135]:
m.head(40)

,plant_id_eia,generator_id,year,fuss,prime_mover,fuel_group,final_ba_code,age_in_current_year,essentials,ba_plus_essentials,age_range,ba_plus_age,essentials_present,ba_plus_essentials_present,ba_plus_age_present,fill_in_score,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,_merge
785664,1,1,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785667,1,2,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785670,1,3,2020,zeroes,IC,petroleum,Alaska,11.000684,2020_IC_petroleum,2020_IC_petroleum_Alaska,2,2020_IC_petroleum_Alaska_2,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
785673,1,5,2020,zeroes,IC,petroleum,Alaska,20.999316,2020_IC_petroleum,2020_IC_petroleum_Alaska,3,2020_IC_petroleum_Alaska_3,1,0,0,1,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.000000e+00,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,both
335705,3,1,2006,fuel_switch,ST,natural_gas,2,67.830253,2006_ST_natural_gas,2006_ST_natural_gas_2,7,2006_ST_natural_gas_2_7,1,1,1,3,2006-01-01,NaN,1.008871e+07,2.951150e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4.120480e+03,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1.123980e+06,NaN,1.120702e+06,3.278278e+03,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,15.915127,13.739177,2.927320,2,<NA>,2,1.050357,13.40423,both
335711,3,1,2007,fuel_switch,ST,natural_gas,2,67.830253,2007_ST_natural_gas,2007_ST_natural_gas_2,7,2007_ST_natural_gas_2_7,1,1,1,3,2007-01-01,NaN,9.807031e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,954656.813326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13.0,13.0,1.028301e+06,NaN,1.028301e+06,NaN,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,52.914442,14.915811,14.738493,2.927320,2,<NA>,2,1.057435,14.461664,both
335717,3,1,2008,fuel_switch,ST,natural_gas,2,67.830253,2008_ST_natural_gas,2008_ST_natural_gas_2,7,2008_ST_natural_gas_2_7,1,1,1,3,2008-01-01,NaN,8.176078e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,865288.470536,NaN,NaN,N

In [137]:
m['fill_in_score'].value_counts(normalize=True)

3    0.386181
2    0.310098
1    0.303721
Name: fill_in_score, dtype: float64

In [144]:
m['fill_in_score'].value_counts()

3    17623
2    14151
1    13860
Name: fill_in_score, dtype: int64

In [140]:
m.query('fill_in_score == 3')['final_ba_code'].unique()

array(['2', 'EPE', 'TVA', '22', 'SWPP', 'APS', 'TEPC', 'SC', 'SEC', 'SRP',
       'WALC', 'ETR', 'MISO', 'JEA', 'CAISO', 'ERCO', 'LDWP', 'PJM',
       'PSCO', 'WACM', '529', 'AEC', 'ISNE', 'FMPP', 'FPL', 'FPC', 'TEC',
       '57', '177', 'DUKE', 'LNT', '210', 'EVRG', 'LGEE', '556', '41',
       '44', '193', '120', '99', '552', 'AECI', '658', 'NEVP', 'PNM',
       'NYIS', '130', 'SCEG', '166', 'PAC', 'AEP', '186', 'PNW', '195',
       '531', 'SOCO', '560', '569'], dtype=object)

In [141]:
m.query('fill_in_score == 3')['age_in_current_year'].unique()

array([67.83025325, 67.41957563, 21.58521561, 72.66803559, 72.41889117,
       41.41820671, 56.50102669, 55.41957563, 61.58521561, 61.41820671,
       60.50102669, 59.50171116, 49.24845996, 35.66872005, 39.00068446,
       59.58658453, 43.50171116, 41.58521561, 40.50102669, 61.50034223,
       61.7522245 , 20.50102669, 66.50239562, 62.66940452, 54.4202601 ,
       63.00068446, 61.16632444, 26.58726899, 37.58521561, 14.91581109,
       13.16632444, 37.83162218, 63.91512663, 64.66803559, 60.66803559,
       69.41820671, 67.50171116, 19.16769336, 42.91581109, 36.1670089 ,
       70.58726899, 53.7522245 , 58.58726899, 19.00068446, 49.58521561,
       34.66940452, 33.41820671, 19.41957563, 36.        , 35.00068446,
       65.41820671, 63.33470226, 48.08213552, 43.08281999, 67.08281999,
       57.33333333, 56.66803559, 67.16769336, 64.41889117, 54.83093771,
       68.        , 62.25051335, 57.08145106, 55.75359343, 69.33333333,
       28.50102669, 64.08213552, 53.33333333, 27.91512663, 19.91

In [175]:
m = self.fill_in_ep_data()

/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/opt/anaconda3/envs/gencost/lib/python3.11/site-packages/etoolbox/utils/pudl_helpers.py:146: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df.loc[date_mask, date_col] = pd.to_datetime(
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/gencost/src/gencost/waterfall.py:258: UserWarning: This crosswalk is not safe, it may have `pf_subplants with multiple primes. Use `safe_xwalk` instead.
  return self._xwalk.grand_crosswalk
/Users/mcastillo/Library/CloudStorage/OneDrive-RMI/Documents/GitHub/genc

In [173]:
m

,plant_id_eia,generator_id,fuss,final_ba_code,age_in_current_year,report_date,biofuel_mmbtu,coal_mmbtu,natural_gas_mmbtu,nuclear_mmbtu,other_mmbtu,other_gas_mmbtu,petroleum_mmbtu,petroleum_coke_mmbtu,renew_mmbtu,biofuel_net_mwh,coal_net_mwh,natural_gas_net_mwh,nuclear_net_mwh,other_net_mwh,other_gas_net_mwh,petroleum_net_mwh,petroleum_coke_net_mwh,renew_net_mwh,generator_starts,fuel_starts,gross_generation_mwh,biofuel_gross_mwh,coal_gross_mwh,natural_gas_gross_mwh,other_gross_mwh,other_gas_gross_mwh,petroleum_gross_mwh,petroleum_coke_gross_mwh,renew_gross_mwh,utility_id_eia,balancing_authority_code_eia,state,capacity_mw,associated_combined_heat_power,duct_burners,bypass_heat_recovery,solid_fuel_gasification,carbon_capture,fluidized_bed_tech,pulverized_coal_tech,stoker_tech,other_combustion_tech,subcritical_tech,supercritical_tech,ultrasupercritical_tech,age_in_report_year,age_of_observation,age_relative_to_prime_avg,pollution_control_costs_per_kw,respondent_id,respondent_id_purchaser,final_respondent_id,wage_scale,age_of_observation_secular_adj,fill_in_match
785664,1,1,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,essentials
785667,1,2,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,essentials
785670,1,3,zeroes,Alaska,11.000684,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,essentials
785673,1,5,zeroes,Alaska,20.999316,2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,343.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.0,138.0,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,20847,MISO,WI,2.7,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,50.836413,1.916496,28.969892,106.666667,193,<NA>,193,0.959081,26.618848,essentials
335705,3,1,fuel_switch,2,67.830253,2006-01-01,NaN,1.008871e+07,29511.495021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,964082.899927,4120.479810,NaN,NaN,NaN,NaN,NaN,NaN,7.0,7.0,1123980.0,NaN,1.120702e+06,3278.277774,NaN,NaN,NaN,NaN,NaN,195,SOCO,AL,153.1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,51.915127,15.915127,13.739177,2.927320,2,<NA>,2,1.050357,13.40423,essentials_ba_age_group
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3996801,64815,GEN2,zeroes,ERCO,2.420260,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436,essentials
3996804,64815,GEN3,zeroes,ERCO,2.420260,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436,essentials
3996807,64816,GEN1,zeroes,ERCO,1.248460,2020-01-01,NaN,NaN,124809.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13447.611111,NaN,NaN,NaN,NaN,NaN,NaN,147.0,149.0,323384.0,NaN,NaN,323384.000000,NaN,NaN,NaN,NaN,NaN,7349,SWPP,TX,9.3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,8.585900,1.916496,-13.280621,NaN,58,<NA>,58,1.013692,27.335436,essentials
3996810,